# P04 · Notebook 22b · Newer encoders: the screen and the fine-tuning that notebook 22 could not run

**Where we start (notebook 22), all at −6 dB:**

| Track | Input | System | Development | Confirmation |
|---|---|---|---|---|
| **ARR** | MWF (8 microphones) | fine-tuned BEATs (notebook 21), scored per frequency band (BEAM) | 0.9794 | 0.9652 |
| **MIC+** | microphone 0, denoised | frozen BEATs, BEAM | 0.8302 | 0.8491 |
| **MIC** | microphone 0 (the DCASE 2020 setting) | fine-tuned BEATs + extra machines + 3 seeds + ECHO (notebook 21) | 0.8377 | 0.9006 |

**Why this notebook:** notebook 22's encoder stages did not run, because none of the new encoders loaded:
- **EAT:** our loader failed under transformers 5, because EAT's own code does not call `post_init`. If `from_pretrained` fails, this notebook builds EAT from its own configuration and code and loads its weights directly.
- **BEATs iter3:** its download links failed. This notebook adds a third source, a copy released by PretrainedSED, read as weights only.

**What it runs (notebook 22's rules, unchanged):**
1. **The screen:** BEATs iter3, EAT-base and EAT-large, frozen, against our BEATs, on each track.
2. **The best new encoder, fine-tuned** with each track's winning recipe. It may replace the track's system, or be fused with it.

**The keep rule:** an idea is kept only if it raises pumps and fans by at least 0.01 and costs valves and slide rails at most 0.01.

**Rules:** `results/mimii_beats/RULES_new_encoders_scoring_written_before_running.md`, with Amendment 1, written before this notebook was built.

**It needs** in Google Drive:
- `P04_BEATs`;
- `P04_TwinFrontEnds` (notebook 19);
- `P04_DenoiseScore` (notebook 20);
- `P04_StrongFineTune` (notebook 21);
- `P04_NewEncoders` (notebook 22's folder; this notebook writes there too).

Drive needs at least 2 GB free. EAT downloads from Hugging Face, and BEATs iter3 from its sources.

**Runtime:**
- Choose **G4** (RTX PRO 6000): Runtime › Change runtime type. Then Runtime › Run all.
- Cell 9b loads and tests every encoder before any MIMII download. Cell 11 prints the training estimate for each candidate encoder.
- Expect about 1.5–3 hours on G4. The selected encoder's 10 trainings take most of it, and EAT-large would take the longest.
- Trainings and embeddings are saved to Drive › `P04_NewEncoders` as soon as they finish. If Colab disconnects, run all again: finished work is skipped, and an unfinished training resumes from its last epoch.

**At the end** `p04_n22b_results.zip` downloads. Send it back here.

In [ ]:
# 1 · Setup
import os, sys, json, time, hashlib, struct, zipfile, subprocess, shutil, urllib.request, re, threading, math, gc, base64, importlib
from pathlib import Path
from collections import Counter, deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from sklearn.cluster import KMeans

TEST = os.environ.get("P04_TEST") == "1"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY = Path("/content/drive/MyDrive")
    OUT, WORK, CKPT_DIR = MY / "P04_NewEncoders", Path("/content/work"), MY / "P04_BEATs"
    N19, N20, N21 = MY / "P04_TwinFrontEnds", MY / "P04_DenoiseScore", MY / "P04_StrongFineTune"
else:
    OUT, WORK = Path(os.environ.get("P04_OUT", "P04_NewEncoders")), Path(os.environ.get("P04_WORK", "work"))
    CKPT_DIR = Path(os.environ.get("P04_CKPT_DIR", str(OUT)))
    N19, N20 = Path(os.environ.get("P04_N19_DIR", "P04_TwinFrontEnds")), Path(os.environ.get("P04_N20_DIR", "P04_DenoiseScore"))
    N21 = Path(os.environ.get("P04_N21_DIR", "P04_StrongFineTune"))
for p in (OUT, WORK, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)
for p, who in ((N19, "notebook 19 (P04_TwinFrontEnds)"), (N20, "notebook 20 (P04_DenoiseScore)"), (N21, "notebook 21 (P04_StrongFineTune)")):
    if not p.exists():
        raise SystemExit(f"missing {p}: the files of {who} must be in Drive")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEVICE == "cuda"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else ""
if DEVICE != "cuda" and not TEST:
    raise SystemExit("No GPU. Runtime › Change runtime type › G4 (or another GPU), then Runtime › Run all.")
T_START = time.time()
DRIVE_FREE_GB = shutil.disk_usage(OUT).free / 1e9
print("device:", DEVICE, GPU, "· torch", torch.__version__, f"· free local disk {shutil.disk_usage(WORK).free / 1e9:.0f} GB"
      f" · free Drive {DRIVE_FREE_GB:.1f} GB")


def md5(path, chunk=8 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sha256(path, chunk=8 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def download(url, dest):
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    try:
        if url.startswith("file://"):
            shutil.copy(url[7:], part)
        elif shutil.which("wget"):
            subprocess.run(["wget", "-q", "--show-progress", "-c", "-O", str(part), url], check=True)
        else:
            urllib.request.urlretrieve(url, part)
        part.rename(dest)
    finally:
        if part.exists() and not dest.exists():
            part.unlink()


download_one = download                              # notebook 11's single-connection download


def download(url, dest):
    # several connections per file with aria2c (notebook 19 saw about 2.4 MB/s per connection from Zenodo); falls back to one connection
    dest = Path(dest); part = dest.with_suffix(dest.suffix + ".part")
    if not url.startswith("file://"):
        if shutil.which("aria2c") is None and IN_COLAB:
            if subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True).returncode != 0:
                subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
                subprocess.run(["apt-get", "-qq", "install", "-y", "aria2"], capture_output=True)
        if shutil.which("aria2c"):
            r = subprocess.run(["aria2c", "-x", "4", "-s", "4", "-k", "8M", "--file-allocation=none", "--max-tries=30", "--retry-wait=10",
                                "--allow-overwrite=true", "--auto-file-renaming=false", "--console-log-level=warn", "--summary-interval=0",
                                "--download-result=hide", "-d", str(dest.parent), "-o", part.name, url])
            if r.returncode == 0 and part.exists():
                part.rename(dest)
                return
            print(f"  aria2c failed for {dest.name} (code {r.returncode}); one connection instead", flush=True)
            for f in (part, Path(str(part) + ".aria2")):
                f.unlink(missing_ok=True)
    download_one(url, dest)

In [ ]:
# 2 · The official BEATs code (the same files as notebook 1)
BEATS_RAW = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"
CODE_SHA256 = {"BEATs.py": "27f289db7c56ce26f2ceb50d3719854b91b2dec1c2830d8b1dd8de1bbee19eeb",
               "backbone.py": "31c0378379a7e0f1d1069f9da444fb86890fe1ea078959a2dcd39640cdcadbaa",
               "modules.py": "edeb6b6cd6a784da749f932c3e0783c0bce556fc768d0d23a4d53d4b819eb424"}
CODE_DIR = WORK / "beats"; CODE_DIR.mkdir(exist_ok=True)
code_sha = {}
for f, want in CODE_SHA256.items():
    p = CODE_DIR / f
    if not p.exists():
        download(BEATS_RAW + f, p)
    code_sha[f] = sha256(p)
    print(f, "same as notebook 1" if code_sha[f] == want else f"CHANGED upstream ({code_sha[f][:12]}…)")
try:
    import torchaudio.compliance.kaldi
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchaudio"], check=True)
sys.path.insert(0, str(CODE_DIR))
from BEATs import BEATs, BEATsConfig

In [ ]:
# 3 · The checkpoint notebook 1 used (from Drive › P04_BEATs); its SHA-256 must match
CKPT_NAME, CKPT_SHA_N1 = "BEATs_iter3_plus_AS2M.pt", "d43cbfad4d7b56381c061d7a24774f908d4d94c72961f6eb1d9090ff18cd8d34"
ckpt = CKPT_DIR / CKPT_NAME
if not ckpt.exists():
    raise SystemExit(f"Put {CKPT_NAME} in Google Drive › MyDrive › P04_BEATs (notebook 1 left it there), then run again.")
CKPT_SHA = sha256(ckpt)
print(CKPT_NAME, "· same file as notebook 1:", CKPT_SHA == CKPT_SHA_N1)
if not TEST:
    assert CKPT_SHA == CKPT_SHA_N1, "stop: not the checkpoint notebook 1 used"
try:
    chk = torch.load(ckpt, map_location="cpu", weights_only=True)
except Exception:
    chk = torch.load(ckpt, map_location="cpu", weights_only=False)
cfg = BEATsConfig(chk["cfg"])
model = BEATs(cfg); model.load_state_dict(chk["model"]); model.eval().to(DEVICE)
print("layers:", cfg.encoder_layers, "· fine-tuned:", cfg.finetuned_model)

In [ ]:
# 4 · Every number below was fixed in the rules before this notebook ran
RULES = "results/mimii_beats/RULES_new_encoders_scoring_written_before_running.md"
AMENDMENT = "Amendment 1 (notebook 22b): stages 3 and 4 with the loaders fixed, from notebook 22's final systems"
PROGRAM = "results/mimii_beats/PROGRAM_full_written_before_running.md"
Z = "https://zenodo.org/records/3384388/files/"
MD5 = {"pump|6dB": "a09ba6060c10fc09cd4c8770213b0b9f", "pump|-6dB": "d20b783a0ff9c93d58f452f98c37b112",
       "fan|6dB": "0890f7d3c2fd8448634e69ff1d66dd47", "fan|-6dB": "f02ae808a58d84b6815b7ec38ff30879",
       "valve|6dB": "fe5fb7c337cd701b1d31dc641e621892", "valve|-6dB": "fdfaf185fea61b21e11952a070a4ada7",
       "slider|6dB": "838c2b3441858359c4704ef13a1b27ff", "slider|-6dB": "49913eda7d37f182cbf8ed5c984140e0"}
ZIPS = {}
for key, h in MD5.items():
    t_, lv_ = key.split("|"); nm = f"{lv_.replace('dB', '_dB')}_{t_}.zip"
    ZIPS[key] = (Z + nm + "?download=1", nm, h)
ZX = "https://zenodo.org/records/3727685/files/"                    # DCASE 2020 Task 2, additional training dataset
XMD5 = {"pump": "4e33ae9c0db5cc88437675f5e317caee", "fan": "567798854130c8019df8b664c095ed1e",
        "valve": "057cdd560caffb8305449541afd28c6d", "slider": "f84f186f4295fc23754abcdd1b0580a7"}
XZIPS = {t_: (ZX + f"eval_data_train_{t_}.zip?download=1", f"eval_data_train_{t_}.zip", h) for t_, h in XMD5.items()}
GB = {"pump": 15.9, "fan": 21.1, "valve": 15.0, "slider": 15.1}     # approximate size of each type's two MIMII zips
if TEST:
    _d = json.loads(os.environ["P04_TEST_DATA"])
    XZIPS = {t_: tuple(v) for t_, v in _d.pop("xc").items()}
    _d.pop("dcase", None)
    ZIPS = {k: tuple(v) for k, v in _d.items() if k.split("|")[1] in ("6dB", "-6dB")}
TYPES, MAIN, AUX = ["pump", "fan", "valve", "slider"], ["pump", "fan"], ["valve", "slider"]
LEVELS = ["6dB", "-6dB"]
SR, L, H = 16000, 1024, 512                                          # notebook 19's beamformer STFT (MWF weights are per bin of it)
STREAMS = ["CH0_m6", "MWF_m6", "MWF_6"]                              # microphone 0 at −6 dB (exact, int16); MWF₋₆ on x₋₆ and on its 6 dB twin (float16)
SHOTS = [1, 3, 10]
SPLITS = ["A", "B", "B+1", "B+3", "B+10"]
FOLDS = {1: ["id_00", "id_02"], 2: ["id_04", "id_06"]}
TRACKS = ["ARR", "MIC+", "MIC"]
ALL_TRACKS = list(TRACKS)                                            # fixed order for random seeds
TRACK_INPUT = {"ARR": "MWF", "MIC+": "N2N-CH0", "MIC": "CH0"}       # notebook 21's inputs (D2′ failed in notebook 22: ARR stays on MWF)
CFG = dict(lora_rank=16, lora_alpha=32, lora_dropout=0.1, emb_dim=256, subcenters=3, arc_scale=30.0, arc_margin=0.2,
           mixup_prob=0.5, mixup_beta=0.5, crop_s=4.0, lr_lora=2e-4, lr_head=1e-3, wd_head=1e-4,
           batch=32, epochs=10, warmup_epochs=1, clip_norm=5.0, xc_cap=600, pl_k=4, pl_starts=10, pl_seed=0,
           ts_prob=0.5, pd_balance_batches=8, se_seeds=[0, 1, 2], embed_batch=32)        # notebook 21's, unchanged
KAPPA = {"6dB": 10 ** -0.6, "-6dB": 10 ** 0.6}                      # (notebook 21's shared code; TS is not used here)
C_SWAP = math.sqrt((KAPPA["-6dB"] - KAPPA["6dB"]) / (KAPPA["-6dB"] + KAPPA["6dB"]))
GT_RANGE, GT_PAIRS = (0.95, 1.05), 200
BAR = 0.01
G0B_TOL, G0C_COS, G0C_CLIPS, G0D_COS = 0.0005, 0.999, 32, 0.999
DRIVE_MIN_GB, LDN_FLOOR = 2.0, 1e-6
FROZEN_REF = {"ARR": (0.9442, 0.8835), "MIC+": (0.8135, 0.8238), "MIC": (0.7633, 0.7767)}   # BEATs LDN1, notebook 20
N21_REF = {"ARR": (0.9672, 0.9558), "MIC+": (0.8135, 0.8238), "MIC": (0.8377, 0.9006)}     # notebook 21's final systems
N22_REF = {"ARR": (0.9794, 0.9652), "MIC+": (0.8302, 0.8491), "MIC": (0.8377, 0.9006)}     # notebook 22's (where 22b starts)
G0E_TOL = 0.0005
PUBLISHED_M6 = {"pump": 0.8977, "fan": 0.8778, "valve": 0.9794, "slider": 0.9441}         # arXiv 2309.07500, Table 1, −6 dB, one microphone
T2_TARGET = {"pump": 0.9785, "fan": 0.9539}                                                # frozen BEATs + BF at 6 dB, B+10 (the program)
N2N_CFG = dict(n_fft=512, hop=256, hidden=256, layers=2, epochs=60, batch=32, lr=1e-3, lr_end=1e-5, clip=5.0, crop_s=4.0,
               val_every=20, norm_crops=200, in_levels=["-6dB"], seed=0)                     # notebook 20's
RDP_GAMMA = 4                                                        # MERL's deviation power
VIEWS = ["mean", "max", f"rdp{RDP_GAMMA}"]
SCORER_VIEWS = {"BEAM": ["mean"], "AdaBEAM": ["mean", "max"], "BEAM-RDP": [f"rdp{RDP_GAMMA}"], "AdaBEAM-RDP": [f"rdp{RDP_GAMMA}", "max"]}
ENCODERS = {"BEATs": {"name": "BEATs iter3+ (AS2M)", "dim": 768, "layers": 12},
            "BEATs-iter3": {"name": "BEATs iter3", "dim": 768, "layers": 12, "file": "BEATs_iter3.pt",
                            "sed_file": "BEATs_iter3_from_PretrainedSED.pt",          # Amendment 1: the third source
                            "sed": "https://github.com/fschmid56/PretrainedSED/releases/download/v0.0.1/BEATs_ssl.pt",
                            "share": "https://1drv.ms/u/s!AqeByhGUtINrgcpxJUNDxg4eU0r-vA?e=qezPJ5",
                            "azure": ("https://valle.blob.core.windows.net/share/BEATs/BEATs_iter3.pt?sv=2020-08-04"
                                      "&st=2023-03-01T07%3A51%3A05Z&se=2033-03-02T07%3A51%3A00Z&sr=c&sp=rl&sig=QJXmSJG9DbMKf48UDIU1MfzIro8HQOf3sqlNXiflY1I%3D")},
            "EAT-base": {"name": "EAT-base (epoch 30, pre-trained)", "dim": 768, "layers": 12, "repo": "worstchan/EAT-base_epoch30_pretrain"},
            "EAT-large": {"name": "EAT-large (epoch 20, pre-trained)", "dim": 1024, "layers": 24, "repo": "worstchan/EAT-large_epoch20_pretrain"}}
NEW_ENCODERS = ["BEATs-iter3", "EAT-base", "EAT-large"]              # the table's order (ties go to the earlier one)
ENC_ORDER = list(ENCODERS)
EAT_NORM, EAT_FRAMES = (-4.268, 4.569), 1024                         # EAT's model card
GB_RATIO = 1.5
ECHO_BANDS = 7                                                       # ECHO-small's utterance output: the CLS of 7 bands × 384
RECIPE = {"ARR": (["PD"], [0], "P"), "MIC": (["XC"], [0, 1, 2], "F"), "MIC+": ([], [0], "P")}   # stage 4 (MIC also fuses ECHO, EF)
if TEST:
    _c = json.loads(os.environ.get("P04_TEST_CFG", "{}"))
    CFG.update(_c.get("cfg", {})); N2N_CFG.update(_c.get("n2n", {}))
    for k_, v_ in _c.get("encoders", {}).items():
        ENCODERS[k_].update(v_)
    if os.environ.get("P04_TEST_TRACKS"):
        TRACKS = os.environ["P04_TEST_TRACKS"].split(",")
print("rules:", RULES, "·", json.dumps(CFG))
if DRIVE_FREE_GB < DRIVE_MIN_GB and not TEST:
    raise SystemExit(f"stop: Drive has {DRIVE_FREE_GB:.1f} GB free and this run needs {DRIVE_MIN_GB} GB. Free some space in Google Drive, then run again.")

In [ ]:
# 5 · Downloads (MD5 against the Zenodo records), a background download of the next type, readers, notebook 11's splits
CHECKED = {}                                        # files whose MD5 was already checked in this session


def get_file(entry):
    url, name, want = entry
    dest = WORK / name
    if CHECKED.get(name) == want and dest.exists():
        return dest
    if dest.exists() and md5(dest) == want:
        CHECKED[name] = want
        return dest
    t = time.time(); download(url, dest)
    got = md5(dest)
    print(f"{name}: downloaded in {time.time() - t:.0f} s · MD5 matches Zenodo: {got == want}", flush=True)
    if got != want:
        dest.unlink()
        raise SystemExit(f"stop: {name} is not the published file")
    CHECKED[name] = want
    return dest


PREFETCH = {}


def prefetch(t):
    # download the next machine type while this one trains its denoisers (only if the disk has room)
    if t is None or t in PREFETCH or (OUT / f"done_{t}.json").exists():
        return
    if shutil.disk_usage(WORK).free / 1e9 < GB.get(t, 0) * 1.05 + 8:
        print(f"  (not enough free disk to download {t} in the background; it will download when its turn comes)")
        return

    def job():
        for lv in LEVELS:
            try:
                get_file(ZIPS[f"{t}|{lv}"])
            except BaseException as e:
                print(f"  (background download of {t} {lv} failed: {repr(e)[:150]}; it will be retried)", flush=True)
    th = threading.Thread(target=job, daemon=True); th.start(); PREFETCH[t] = th
    print(f"  background download of {t} started")


def read_all(data):
    # WAV bytes (plain or WAVE_FORMAT_EXTENSIBLE, 16-bit PCM) -> (samples, channels) float32 in [-1, 1], sample rate
    assert data[:4] == b"RIFF" and data[8:12] == b"WAVE", "not a WAV file"
    pos, fmt, raw = 12, None, None
    while pos + 8 <= len(data):
        cid, size = data[pos:pos + 4], struct.unpack("<I", data[pos + 4:pos + 8])[0]
        body = data[pos + 8:pos + 8 + size]
        if cid == b"fmt ":
            tag, ch, sr, _, _, bits = struct.unpack("<HHIIHH", body[:16])
            if tag == 0xFFFE:
                tag = struct.unpack("<H", body[24:26])[0]
            fmt = (tag, ch, sr, bits)
        elif cid == b"data":
            raw = body
        pos += 8 + size + (size & 1)
    tag, ch, sr, bits = fmt
    assert tag == 1 and bits == 16, f"expected 16-bit PCM, got format {tag}, {bits}-bit"
    nf = len(raw) // (2 * ch)
    return np.frombuffer(raw[:nf * 2 * ch], dtype="<i2").reshape(-1, ch).astype(np.float32) / 32768.0, sr


def parse(names):
    pump = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    label = np.array([1 if n.replace("\\", "/").split("/")[-2] == "abnormal" else 0 for n in names])
    return pump, label


def split_a(pump, label, fname, p):
    # the same split as notebooks 1 and 3: test = first N normal by name + all abnormal of pump p; train = the other normal
    idx = np.where(pump == p)[0]
    nor = idx[label[idx] == 0]; abn = idx[label[idx] == 1]
    nor = nor[np.argsort(fname[nor])]
    return nor[len(abn):], np.r_[nor[:len(abn)], abn]



STORE = WORK / "stores22"; STORE.mkdir(exist_ok=True)
FLOAT_STREAMS = {"MWF_m6", "MWF_6"}                                  # processed signals are kept in float16, never clipped


def ready(t):
    # the type's streams are in this session's local store
    return (STORE / f"{t}.json").exists()


def prefetch(t):
    # download the next type's two zips in the background while this one is prepared (only if the disk has room)
    if t is None or t in PREFETCH or ready(t):
        return
    if shutil.disk_usage(WORK).free / 1e9 < GB.get(t, 0) * 1.05 + 8:
        print(f"  (not enough free disk to download {t} in the background; it will download when its turn comes)")
        return

    def job():
        for lv in LEVELS:
            try:
                get_file(ZIPS[f"{t}|{lv}"])
            except BaseException as e:
                print(f"  (background download of {t} {lv} failed: {repr(e)[:150]}; it will be retried)", flush=True)
    th = threading.Thread(target=job, daemon=True); th.start(); PREFETCH[t] = th
    print(f"  background download of {t} started")


def open2(t):
    return {lv: zipfile.ZipFile(get_file(ZIPS[f"{t}|{lv}"])) for lv in LEVELS}


def read2(zfs, name):
    # one clip at 6 dB and at −6 dB (its twins): level → (samples, 8) float32
    out = {}
    for lv, zf in zfs.items():
        x, sr = read_all(zf.read(name))
        assert sr == SR and x.ndim == 2 and x.shape[1] == 8, (lv, name, sr, x.shape)
        if not TEST:
            assert x.shape[0] == 10 * SR, (lv, name, x.shape)
        out[lv] = x
    return out


def stream(fn, items, depth=8, workers=2):
    # fn(item) runs in background threads, results come back in order: unzipping overlaps the main thread's work
    with ThreadPoolExecutor(workers) as ex:
        q, it = deque(), iter(items)
        for item in it:
            q.append(ex.submit(fn, item))
            if len(q) >= depth:
                break
        while q:
            res = q.popleft().result()
            for item in it:
                q.append(ex.submit(fn, item))
                break
            yield res

In [ ]:
# 6 · The STFT and beamforming of notebook 19 (MWF is applied with its saved weights), and a reconstruction check
SQH = torch.sqrt(torch.hann_window(L, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft(x):
    # (channels, samples) float32 tensor → (channels, bins, frames) complex64; centred, zero padding, square-root Hann
    return torch.stft(x, n_fft=L, hop_length=H, window=SQH, center=True, pad_mode="constant", return_complex=True)


def istft(Y, n):
    return torch.istft(Y, n_fft=L, hop_length=H, window=SQH, center=True, length=n)


def to_dev(x):
    # (samples, channels) numpy → (channels, samples) tensor on the device
    return torch.from_numpy(np.ascontiguousarray(x.T)).to(DEVICE)


def beam(w, X):
    # w: (bins, 8) complex64 tensor, X: (8, bins, frames) → wᴴx: (bins, frames)
    return torch.einsum("km,mkt->kt", w.conj(), X)



_x = (np.random.default_rng(0).standard_normal((3 * SR, 8)) * 0.1).astype(np.float32)
_pr = float(np.abs(istft(stft(to_dev(_x)), len(_x)).cpu().numpy().T - _x).max())
print(f"STFT reconstruction error {_pr:.1e}")
assert _pr < 1e-4

In [ ]:
# 7 · Noise2Noise post-filter (notebook 19's cell, unchanged; it applies notebook 20's denoisers): the network, the training loop, inference, and a smoke test on synthetic twins
NF, NH = N2N_CFG["n_fft"], N2N_CFG["hop"]
NB = NF // 2 + 1
SQH2 = torch.sqrt(torch.hann_window(NF, periodic=True, dtype=torch.float32)).to(DEVICE)


def stft2(x):
    return torch.stft(x, n_fft=NF, hop_length=NH, window=SQH2, center=True, pad_mode="constant", return_complex=True)


def istft2(Y, n):
    return torch.istft(Y, n_fft=NF, hop_length=NH, window=SQH2, center=True, length=n)


class N2NNet(nn.Module):
    # log-power per bin → Linear → ReLU → 2-layer bidirectional GRU → Linear → sigmoid: a real mask per bin and frame
    def __init__(self):
        super().__init__()
        self.register_buffer("mu", torch.zeros(NB)); self.register_buffer("sd", torch.ones(NB))
        self.inp = nn.Linear(NB, N2N_CFG["hidden"])
        self.rnn = nn.GRU(N2N_CFG["hidden"], N2N_CFG["hidden"], num_layers=N2N_CFG["layers"], batch_first=True, bidirectional=True)
        self.out = nn.Linear(2 * N2N_CFG["hidden"], NB)

    def forward(self, Y):
        f = torch.log10(Y.real ** 2 + Y.imag ** 2 + 1e-10).transpose(1, 2)
        z, _ = self.rnn(torch.relu(self.inp((f - self.mu) / self.sd)))
        return torch.sigmoid(self.out(z)).transpose(1, 2)


def n2n_loss(M, Y, T):
    # per sample Σ|M·Y − T|² / Σ|Y|² (the weight depends on the input only, so the Noise2Noise optimum is unchanged)
    num = ((M * Y - T).abs() ** 2).sum((1, 2)); den = (Y.abs() ** 2).sum((1, 2)).clamp_min(1e-12)
    return (num / den).mean()


def val_loss(net, vx, vt):
    net.eval(); tot = idl = 0.0
    with torch.no_grad():
        for i in range(0, len(vx), 32):
            Y, T = stft2(vx[i:i + 32]), stft2(vt[i:i + 32]); M = net(Y)
            tot += float(n2n_loss(M, Y, T)) * len(Y); idl += float(n2n_loss(torch.ones_like(M), Y, T)) * len(Y)
    net.train()
    return tot / len(vx), idl / len(vx)


def fit(get, n_pairs, norm_x, vx, vt, seed, epochs=None):
    # get(index array, rng) → (inputs, targets) float32 tensors on the device; each epoch is one pass over a fresh permutation
    epochs = epochs or N2N_CFG["epochs"]
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = N2NNet().to(DEVICE)
    with torch.no_grad():
        Fx = torch.cat([torch.log10(stft2(c).abs() ** 2 + 1e-10).permute(0, 2, 1).reshape(-1, NB) for c in norm_x])
        net.mu.copy_(Fx.mean(0)); net.sd.copy_(Fx.std(0).clamp_min(1e-3))
    B = N2N_CFG["batch"]; steps = max(1, math.ceil(n_pairs / B))
    opt = torch.optim.Adam(net.parameters(), lr=N2N_CFG["lr"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps * epochs, eta_min=N2N_CFG["lr_end"])
    v0, vi0 = val_loss(net, vx, vt)
    log = [{"epoch": 0, "train": None, "val": round(v0, 6), "val_identity": round(vi0, 6), "seconds": 0.0}]
    for ep in range(epochs):
        t0 = time.time(); order = rng.permutation(n_pairs); run = 0.0
        for b in range(steps):
            x, tg = get(order[b * B:(b + 1) * B], rng)
            Y, T = stft2(x), stft2(tg)
            loss = n2n_loss(net(Y), Y, T)
            opt.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), N2N_CFG["clip"]); opt.step(); sched.step()
            run += float(loss.detach())
        v, vi = val_loss(net, vx, vt)
        log.append({"epoch": ep + 1, "train": round(run / steps, 6), "val": round(v, 6), "val_identity": round(vi, 6), "seconds": round(time.time() - t0, 1)})
    return net, log


@torch.no_grad()
def denoise(net, x):
    # x: (batch, samples) float32 tensor → the post-filtered signal
    Y = stft2(x)
    return istft2(net(Y) * Y, x.shape[1])


# smoke test: harmonic sounds with independent noise in input and target; the denoiser must beat doing nothing on held-out clips
rng = np.random.default_rng(1)
ns, nc = 16000, 96
tt = torch.arange(ns, dtype=torch.float32) / SR
clean = torch.stack([sum(torch.sin(2 * math.pi * float(f0) * k * tt + float(rng.uniform(0, 6))) / k for k in range(1, 9))
                     for f0 in rng.uniform(80, 400, nc)]).to(DEVICE)
nz = lambda g: torch.from_numpy(rng.standard_normal((nc, ns)).astype(np.float32)).to(DEVICE) * g
xin, xtg = clean + nz(2.0), clean + nz(1.0)
net_s, log_s = fit(lambda idx, r: (xin[idx], xtg[idx]), 80, [xin[:16]], xin[80:], xtg[80:], seed=0, epochs=15 if not TEST else 6)
v0, v, vi = log_s[0]["val"], log_s[-1]["val"], log_s[-1]["val_identity"]
print(f"N2N smoke test: validation loss {v0:.3f} untrained → {v:.3f} after {len(log_s) - 1} epochs · {vi:.3f} for doing nothing")
assert v < vi and v < v0, "smoke test: the Noise2Noise denoiser does not learn"
del net_s

In [ ]:
# 8 · Model pieces (notebook 2's code): our forward pass, LoRA, the head and the sub-center ArcFace layer
def encode(x, amp):
    """Waveforms (B, N) in [-1, 1] → mean over time of the last layer (B, 768), float32.
    Same steps as BEATs.extract_features without padding; the filter banks are always computed in float32."""
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # (B, frames, 128)
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float().mean(1)


class LoRALinear(nn.Module):
    """y = W x + (α / r) · B A x. W stays frozen; B starts at zero, so the model starts exactly as the frozen one."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(base.out_features, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale


class Head(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out, bias=False)
        self.bn = nn.BatchNorm1d(d_out)

    def forward(self, x):
        return self.bn(self.lin(x))


class SubCenterArcFace(nn.Module):
    """Cosine to the closest of k centres per class; additive angular margin m on the true class; scale s."""
    def __init__(self, d, n_classes, k, s, m):
        super().__init__()
        self.W = nn.Parameter(torch.empty(n_classes * k, d))
        nn.init.xavier_uniform_(self.W)
        self.n, self.k, self.s, self.m = n_classes, k, s, m
        self.cos_m, self.sin_m = math.cos(m), math.sin(m)
        self.th, self.mm = math.cos(math.pi - m), math.sin(math.pi - m) * m

    def cosine(self, e):
        c = F.linear(F.normalize(e.float(), dim=1), F.normalize(self.W.float(), dim=1))
        return c.view(-1, self.n, self.k).max(dim=2).values

    def loss(self, cos, y):
        sin = torch.sqrt((1.0 - cos * cos).clamp(0.0, 1.0))
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)      # keeps the target logit monotonic near θ = π
        one = F.one_hot(y, self.n).to(cos.dtype)
        return F.cross_entropy(self.s * (one * phi + (1.0 - one) * cos), y)


def equal_length_batches(clips, idx, batch):
    """Consecutive clips of the same length, at most `batch` per group (the same grouping as notebook 1)."""
    buf = []
    for i in idx:
        if buf and len(clips[i]) != len(clips[buf[0]]):
            yield buf; buf = []
        buf.append(i)
        if len(buf) == batch:
            yield buf; buf = []
    if buf:
        yield buf


def as_float(clips, ids):
    return torch.from_numpy(np.stack([clips[i] for i in ids]).astype(np.float32) / 32768.0).to(DEVICE)

In [ ]:
# 9 · LoRA on the query and value projections of all 12 layers, with one switch (PD's frozen teacher runs with it off); checks; a timed step
for p_ in model.parameters():
    p_.requires_grad_(False)                                          # the original weights never change


class LoRASwitch(LoRALinear):
    on = True                                                         # one switch for every block

    def forward(self, x):
        y = self.base(x)
        return y + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale if LoRASwitch.on else y


def tokens(x, amp):
    # waveforms (B, N) in [-1, 1] → last-layer tokens (B, T, 768), float32: notebook 2's encode without the final mean
    with torch.autocast(device_type=DEVICE, enabled=False):
        fb = model.preprocess(x.float())                              # filter banks always in float32
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = model.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = model.layer_norm(f)
        if model.post_extract_proj is not None:
            f = model.post_extract_proj(f)
        h, _ = model.encoder(model.dropout_input(f), padding_mask=None)
    return h.float()


rng0 = np.random.default_rng(0)
CROP = int(CFG["crop_s"] * SR)
x4 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (4, CROP)).astype(np.float32)).to(DEVICE)
with torch.no_grad():
    Z4 = tokens(x4, amp=False).mean(1)
    ref = model.extract_features(x4)[0].mean(1)
    gap_e = float((encode(x4, amp=False) - Z4).abs().max())
gap = float((Z4 - ref).abs().max())
print(f"our forward pass against BEATs.extract_features: largest difference {gap:.2e} (notebook 2's encode: {gap_e:.1e})")
assert gap < 1e-3 and gap_e < 1e-5, "stop: our forward pass does not reproduce BEATs"
LORA = []
for layer in model.encoder.layers:
    att = layer.self_attn
    for name in ("q_proj", "v_proj"):
        w = LoRASwitch(getattr(att, name), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"])
        setattr(att, name, w); LORA.append(w)
with torch.no_grad():
    gap = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
print(len(LORA), "LoRA blocks · with LoRA at zero, largest difference from frozen BEATs:", f"{gap:.2e}")
assert gap < 1e-5, "stop: adding LoRA changed the model before training"


def lora_params():
    return [p for w in LORA for p in (w.A, w.B)]


def fresh_adapters(seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen BEATs
    torch.manual_seed(seed)
    for w in LORA:
        w.reset()
    head = Head(768, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


head_t, arc_t = fresh_adapters(12345, 8)
opt_t = torch.optim.AdamW(lora_params() + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
model.train(); head_t.train(); arc_t.train(); times = []
for k in range(3):
    xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
    yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    loss = arc_t.loss(arc_t.cosine(head_t(tokens(xb, amp=AMP).mean(1))), yb)
    opt_t.zero_grad(set_to_none=True); loss.backward()
    if k == 0:
        assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in LORA if w.B.grad is not None) > 0, "stop: no gradient reaches LoRA"
    opt_t.step()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    times.append(time.time() - t0)
model.eval()
with torch.no_grad():
    moved = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = False
    off = float((tokens(x4, amp=False).mean(1) - Z4).abs().max())
    LoRASwitch.on = True
print(f"after 3 steps: with LoRA on the output moved by {moved:.2e}; with the switch off it differs from frozen BEATs by {off:.2e}")
assert moved > 1e-6 and off < 1e-5, "stop: the LoRA switch does not work"
del head_t, arc_t, opt_t
for w in LORA:
    w.reset()
STEP_S = min(times[1:])
print(f"BEATs iter3+ (AS2M): one training step {STEP_S:.3f} s")

In [ ]:
# 9b · The other encoders (BEATs iter3, EAT-base, EAT-large), with Amendment 1's loaders: load, smoke test, LoRA, GB, a timed step
import torchaudio.compliance.kaldi as ta_kaldi


def beats_fb_tokens(m, fb, amp):
    # filter banks (B, frames, 128) of any BEATs model → last-layer tokens (B, T·8, 768): the forward pass of cell 9
    with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
        f = m.patch_embedding(fb.unsqueeze(1))
        f = f.reshape(f.shape[0], f.shape[1], -1).transpose(1, 2)
        f = m.layer_norm(f)
        if m.post_extract_proj is not None:
            f = m.post_extract_proj(f)
        h, _ = m.encoder(m.dropout_input(f), padding_mask=None)
    return h.float()


def kaldi_frames(n):
    # frames of a 25 ms / 10 ms Kaldi filter bank of n samples (edges snipped)
    return 1 + (n - 400) // 160


class LoRAQKV(nn.Module):
    """A fused query-key-value projection with LoRA on the query and value parts only: y = W x + (α / r) · [B_q A_q x, 0, B_v A_v x].
    B starts at zero, so the model starts exactly as the frozen one; LoRASwitch.on switches every block at once."""
    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        d = base.in_features
        assert base.out_features == 3 * d, (base.in_features, base.out_features)
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(2, r, d, device=base.weight.device))
        self.B = nn.Parameter(torch.empty(2, d, r, device=base.weight.device))
        self.drop = nn.Dropout(dropout)
        self.reset()

    def reset(self):
        with torch.no_grad():
            for i in range(2):
                nn.init.kaiming_uniform_(self.A[i], a=math.sqrt(5))
            self.B.zero_()

    @property
    def weight(self):
        return self.base.weight

    @property
    def bias(self):
        return self.base.bias

    def forward(self, x):
        y = self.base(x)
        if not LoRASwitch.on:
            return y
        h = self.drop(x)
        dq = F.linear(F.linear(h, self.A[0]), self.B[0]) * self.scale
        dv = F.linear(F.linear(h, self.A[1]), self.B[1]) * self.scale
        return y + torch.cat([dq, torch.zeros_like(dq), dv], -1)


class Enc:
    # one encoder: waveforms → last-layer patch tokens (no CLS), its LoRA blocks, its patch layout, and pooling over the time
    # patches that start inside the clip
    def __init__(self, key, net, lora, info=None):
        self.key, self.name, self.dim, self.net, self.lora = key, ENCODERS[key]["name"], ENCODERS[key]["dim"], net, list(lora)
        self.layout, self.ok_lora, self.step_s, self.info = "frequency fastest", bool(lora), None, dict(info or {})

    def tokens(self, x, amp):
        with torch.autocast(device_type=DEVICE, enabled=False):
            fb = self.wave2fb(x.float())                              # filter banks always in float32
        return self.fb2tok(fb, amp)

    def grid(self, h, n_samples):
        # (B, L, D) tokens → (B, T, 8, D): the time patches that start inside a clip of n_samples, in this encoder's layout
        B, Lt, D = h.shape; Tp = Lt // 8
        g = h.view(B, Tp, 8, D) if self.layout == "frequency fastest" else h.view(B, 8, Tp, D).transpose(1, 2)
        return g[:, :max(1, min(Tp, -(-kaldi_frames(n_samples) // 16)))]

    def pool(self, h, n_samples):
        return self.grid(h, n_samples).mean((1, 2))

    def train(self):
        self.net.train()

    def eval(self):
        self.net.eval()

    def reset_lora(self):
        for w in self.lora:
            w.reset()


class BeatsEnc(Enc):
    def wave2fb(self, x):
        return self.net.preprocess(x)

    def fb2tok(self, fb, amp):
        return beats_fb_tokens(self.net, fb, amp)


class EatEnc(Enc):
    varlen = False

    def wave2fb(self, x):
        # EAT's model card: waveform minus its mean; Kaldi filter banks (128 mel, Hanning, 10 ms shift, no dither, HTK-compatible);
        # zero-padded to 1,024 frames (training crops: to a multiple of 16 frames when the model accepts it); then normalised
        out = []
        for w in x:
            w = w - w.mean()
            out.append(ta_kaldi.fbank(w.unsqueeze(0), htk_compat=True, sample_frequency=SR, use_energy=False, window_type="hanning",
                                      num_mel_bins=128, dither=0.0, frame_shift=10))
        fb = torch.stack(out); n = fb.shape[1]
        tgt = EAT_FRAMES if (n > EAT_FRAMES - 128 or not self.varlen) else 16 * math.ceil(n / 16)
        fb = F.pad(fb, (0, 0, 0, tgt - n)) if tgt >= n else fb[:, :tgt]
        return (fb - EAT_NORM[0]) / (2.0 * EAT_NORM[1])

    def fb2tok(self, fb, amp):
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=amp):
            out = self.net.extract_features(fb.unsqueeze(1))
        if isinstance(out, dict):
            out = out["x"] if "x" in out else next(iter(out.values()))
        if isinstance(out, (tuple, list)):
            out = out[0]
        return out[:, 1:].float()                                     # without the CLS token


class StandInAttn(nn.Module):
    # TEST only: attention with one fused qkv projection, like EAT's
    def __init__(self, dim, heads=2):
        super().__init__()
        self.h, self.qkv, self.proj = heads, nn.Linear(dim, 3 * dim), nn.Linear(dim, dim)

    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).view(B, N, 3, self.h, C // self.h).permute(2, 0, 3, 1, 4)
        return self.proj(F.scaled_dot_product_attention(q, k, v).transpose(1, 2).reshape(B, N, C))


class StandInBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.n1, self.attn, self.n2 = nn.LayerNorm(dim), StandInAttn(dim), nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Linear(2 * dim, dim))

    def forward(self, x):
        x = x + self.attn(self.n1(x))
        return x + self.mlp(self.n2(x))


class EatStandIn(nn.Module):
    # TEST only: a small ViT with EAT's interface: extract_features([B, 1, T, 128]) → CLS + (T/16 × 8) patch tokens
    def __init__(self, dim, layers):
        super().__init__()
        self.patch, self.cls = nn.Conv2d(1, dim, 16, 16), nn.Parameter(torch.zeros(1, 1, dim))
        self.pos = nn.Parameter(torch.randn(1, 64 * 8, dim) * 0.02)
        self.blocks, self.norm = nn.ModuleList(StandInBlock(dim) for _ in range(layers)), nn.LayerNorm(dim)

    def extract_features(self, mel):
        x = self.patch(mel).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(x.shape[0], -1, -1), x + self.pos[:, :x.shape[1]]], 1)
        for b in self.blocks:
            x = b(x)
        return self.norm(x)


def onedrive_direct(share_url):
    # OneDrive share link → direct download (the documented 'shares' form of the OneDrive API), as notebook 1
    b64 = base64.b64encode(share_url.encode()).decode().rstrip("=").replace("/", "_").replace("+", "-")
    return f"https://api.onedrive.com/v1.0/shares/u!{b64}/root/content"


def attach_beats_lora(m):
    lora = []
    for layer in m.encoder.layers:
        att = layer.self_attn
        for nm in ("q_proj", "v_proj"):
            w = LoRASwitch(getattr(att, nm), CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(att, nm, w); lora.append(w)
    return lora


WN = ((".parametrizations.weight.original0", ".weight_g"), (".parametrizations.weight.original1", ".weight_v"))


def tensors_of(ck):
    # the weights of a loaded checkpoint (a bare state dict, or under "model" or "state_dict") and its configuration, if it has one
    if isinstance(ck, dict):
        for k in ("model", "state_dict"):
            if isinstance(ck.get(k), dict):
                return ck[k], ck.get("cfg")
        if ck and all(torch.is_tensor(v) for v in ck.values()):
            return ck, None
    raise RuntimeError(f"no weights in the file (it holds a {type(ck).__name__}" + (f" with keys {list(ck)[:5]})" if isinstance(ck, dict) else ")"))


def wn_names(sd, own_keys):
    # torch names weight norm's two tensors in two ways (weight_g / weight_v, and parametrizations); the file's keys take the model's
    old = any(k.endswith(".weight_g") for k in own_keys)
    out = {}
    for k, v in sd.items():
        for a, b in WN:
            k = k.replace(a, b) if old else k.replace(b, a)
        out[k] = v
    return out


def match_weights(net, sd):
    # Amendment 1: the file's keys on the model's own, as they are or with one leading prefix removed or added (whichever matches
    # the most keys). Every trainable parameter must come from the file; fixed tensors (buffers, and parameters without gradient)
    # may come from the model's own construction; keys of the file that the model does not have are not used. Both are listed.
    own = net.state_dict(); keys = set(own)
    trainable = {n for n, q in net.named_parameters() if q.requires_grad}
    sd = wn_names(sd, keys)
    cands = {("as is", "")}
    for k in sd:
        for o in keys:
            if k.endswith("." + o):
                cands.add(("removed", k[:len(k) - len(o)]))
            elif o.endswith("." + k):
                cands.add(("added", o[:len(o) - len(k)]))

    def mapping(how, p):
        return {x[len(p):]: x for x in sd if x.startswith(p)} if how == "removed" else {p + x: x for x in sd}

    how, p = max(sorted(cands), key=lambda c: (len(set(mapping(*c)) & keys), c[0] == "as is"))
    m = {k: x for k, x in mapping(how, p).items() if k in keys}
    other = [k for k in m if tuple(sd[m[k]].shape) != tuple(own[k].shape)]
    missing = sorted(keys - set(m)); lost = [k for k in missing if k in trainable]
    unused = sorted(set(sd) - set(m.values()))
    if other or lost:
        raise RuntimeError(f"the file's weights do not fit the model: {len(lost)} trainable parameters are not in the file (e.g. {lost[:3]}), "
                           f"{len(other)} have another shape (e.g. {other[:3]})")
    rep = {"keys_loaded": len(m), "key_prefix": f"'{p}' {how}" if p else "none", "fixed_tensors_from_construction": missing,
           "file_keys_not_used": len(unused), "file_keys_not_used_e_g": unused[:5]}
    return {k: sd[x] for k, x in m.items()}, rep


def load_matched(net, sd):
    state, rep = match_weights(net, sd)
    res = net.load_state_dict(state, strict=False)
    assert not res.unexpected_keys and sorted(res.missing_keys) == rep["fixed_tensors_from_construction"], res
    return rep


def beats_iter3_file():
    # a BEATs iter3 file already in Drive is used as it is; otherwise the authors' link, the Azure copy, then PretrainedSED's copy
    spec = ENCODERS["BEATs-iter3"]; d = WORK if TEST else CKPT_DIR
    own, sed = d / spec["file"], d / spec["sed_file"]
    for f in (own, sed):
        if f.exists() and f.stat().st_size < 10_000_000:
            f.unlink()                                                # an error page saved by a failed attempt, not the model
    if not (own.exists() or sed.exists()):
        for label, url, dest in (("the authors' link (OneDrive)", spec["share"] and onedrive_direct(spec["share"]), own),
                                 ("Microsoft's Azure copy", spec["azure"], own), ("PretrainedSED's release v0.0.1", spec["sed"], sed)):
            if not url:
                continue
            try:
                download(url, dest)
            except Exception as e:
                print(f"BEATs iter3 · {label}: failed ({e.__class__.__name__})")
            if dest.exists() and dest.stat().st_size < 10_000_000:
                dest.unlink()
            if dest.exists():
                print(f"BEATs iter3 · {label}: downloaded", flush=True)
                break
    for f, label in ((own, "the authors' file"), (sed, "PretrainedSED's copy (release v0.0.1); provenance caveat in Amendment 1")):
        if f.exists():
            return f, label
    raise RuntimeError("no download of BEATs iter3 worked (the authors' link, the Azure copy, PretrainedSED's copy)")


def load_beats_iter3():
    f, label = beats_iter3_file()
    ck = torch.load(f, map_location="cpu", weights_only=True)          # tensors and plain values only: no code in the file runs
    sd, cfg_f = tensors_of(ck)
    cfg_ = cfg_f if cfg_f is not None else {**chk["cfg"], "finetuned_model": False}   # weights only: our iter3+ architecture, no classifier
    m = BEATs(BEATsConfig(cfg_))
    rep = load_matched(m, sd)
    msd, ref = m.state_dict(), chk["model"]
    com = [k for k in msd if k in ref and tuple(ref[k].shape) == tuple(msd[k].shape)]
    dist = math.sqrt(sum(float((msd[k].double() - ref[k].double()).pow(2).sum()) for k in com)
                     / max(sum(float(ref[k].double().pow(2).sum()) for k in com), 1e-30))
    m.eval().to(DEVICE)
    for q in m.parameters():
        q.requires_grad_(False)
    info = {"checkpoint_sha256": sha256(f), "file": f.name, "source": label,
            "configuration": "the file's own" if cfg_f is not None else "our iter3+ checkpoint's, classifier off (the file holds weights only)",
            "fine_tuned_on_audioset": bool(getattr(BEATsConfig(cfg_), "finetuned_model", False)),
            "relative_distance_to_our_iter3plus": round(dist, 4), "tensors_compared": len(com), **rep}
    print(f"BEATs iter3 · {label} · SHA-256 {info['checkpoint_sha256'][:16]}… · relative distance from our iter3+ weights {dist:.4f}")
    return BeatsEnc("BEATs-iter3", m, attach_beats_lora(m), info)


def eat_snapshot(repo):
    # the repository's files at one commit: configuration, code and weights (TEST: a local folder)
    if os.path.isdir(repo):
        return repo, "a local folder (TEST)"
    from huggingface_hub import HfApi, snapshot_download
    names = HfApi().list_repo_files(repo)
    pats = ["*.json", "*.py", "*.txt"] + (["*.safetensors"] if any(n.endswith(".safetensors") for n in names) else ["*.bin"])
    path = snapshot_download(repo, allow_patterns=pats)
    return path, Path(path).name                                      # the snapshot folder is named by its commit


def read_weights(path):
    # every weights file of the snapshot: safetensors, else pytorch_model*.bin read with weights_only=True
    files, sd = sorted(Path(path).glob("*.safetensors")), {}
    if files:
        from safetensors.torch import load_file
        for f in files:
            sd.update(load_file(str(f)))
    else:
        files = sorted(Path(path).glob("pytorch_model*.bin"))
        for f in files:
            sd.update(torch.load(f, map_location="cpu", weights_only=True))
    if not sd:
        raise RuntimeError("no weights file (safetensors or pytorch_model.bin) in the repository")
    return sd, [f.name for f in files]


def load_remote_model(path):
    # from_pretrained first; if it fails (transformers 5 with remote code that does not call post_init), the model is built from
    # the repository's own configuration and code, and its weights file is loaded into it (Amendment 1). A package that its
    # code needs is installed, then the load is tried again.
    from transformers import AutoConfig, AutoModel
    from transformers.dynamic_module_utils import get_class_from_dynamic_module
    for _ in range(4):
        try:
            try:
                return AutoModel.from_pretrained(path, trust_remote_code=True), {"loaded_by": "from_pretrained"}
            except ImportError:
                raise
            except Exception as e:
                first = repr(e)[:300]
            print(f"  from_pretrained failed ({first[:140]}); building the model from its own configuration and code", flush=True)
            conf = AutoConfig.from_pretrained(path, trust_remote_code=True)
            net = get_class_from_dynamic_module(conf.auto_map["AutoModel"], path)(conf)
            sd, files = read_weights(path)
            rep = load_matched(net, sd)
            return net, {"loaded_by": "built from its own configuration and code; weights loaded directly", "from_pretrained_error": first,
                         "weights_files": files, **rep}
        except ImportError as e:
            if isinstance(e, ModuleNotFoundError) and e.name:
                need = [e.name.split(".")[0]]
            else:
                need = " ".join(re.findall(r"pip install ([^`\n]+)", str(e))[:1]).split()
            if not need:
                raise
            print(f"  its code needs {need}; installing", flush=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)
            importlib.invalidate_caches()
    raise RuntimeError("did not load")


def load_eat(key):
    spec = ENCODERS[key]
    if TEST and not os.path.isdir(str(spec.get("repo"))):
        torch.manual_seed(ENC_ORDER.index(key)); net, info = EatStandIn(spec["dim"], spec["layers"]), {"commit": "TEST stand-in"}
    else:
        try:
            import transformers
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers"], check=True)
            import transformers
        path, commit = eat_snapshot(spec["repo"])
        net, info = load_remote_model(path)
        info.update(commit=commit, transformers=transformers.__version__)
    net = net.float().to(DEVICE).eval()
    for q in net.parameters():
        q.requires_grad_(False)
    info = {"repo": spec.get("repo"), **info, "parameters_millions": round(sum(q.numel() for q in net.parameters()) / 1e6, 1)}
    fused = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] == "qkv"]
    split = [(n, m) for n, m in net.named_modules() if isinstance(m, nn.Linear) and n.split(".")[-1] in ("q_proj", "v_proj", "query", "value")]
    lora = []
    if len(fused) == spec["layers"]:
        for n, m in fused:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRAQKV(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(fused)} fused query-key-value projections (query and value parts)"
    elif len(split) == 2 * spec["layers"]:
        for n, m in split:
            parent = net.get_submodule(n.rsplit(".", 1)[0]) if "." in n else net
            w = LoRASwitch(m, CFG["lora_rank"], CFG["lora_alpha"], CFG["lora_dropout"]); setattr(parent, n.split(".")[-1], w); lora.append(w)
        info["lora_on"] = f"{len(split)} separate query and value projections"
    else:
        info["lora_problem"] = f"found {len(fused)} fused qkv and {len(split)} separate query/value projections; {spec['layers']} layers expected"
    return EatEnc(key, net, lora, info)


def lora_params_of(enc):
    return [p for w in enc.lora for p in (w.A, w.B)]


def smoke(enc, timed=True):
    # tokens: shape, finiteness; EAT: other lengths; LoRA: unchanged at zero, a gradient reaches it, the switch works; a timed step
    rep = {}
    x10 = torch.from_numpy(rng0.uniform(-0.1, 0.1, (2, 10 * SR)).astype(np.float32)).to(DEVICE)
    with torch.no_grad():
        h = enc.tokens(x10, amp=False)
    tp = EAT_FRAMES // 16 if isinstance(enc, EatEnc) else kaldi_frames(10 * SR) // 16
    rep["tokens_of_10_s"] = list(h.shape)
    if not (h.shape[1] == tp * 8 and h.shape[2] == enc.dim and bool(torch.isfinite(h).all())):
        raise RuntimeError(f"{enc.key}: tokens {tuple(h.shape)}, expected (2, {tp * 8}, {enc.dim}) and finite")
    if isinstance(enc, EatEnc):
        enc.varlen = True
        try:
            with torch.no_grad():
                hc = enc.tokens(x10[:, :CROP], amp=False)
            enc.varlen = bool(hc.shape[1] == 16 * math.ceil(kaldi_frames(CROP) / 16) // 16 * 8 and torch.isfinite(hc).all())
        except Exception as e:
            enc.varlen = False; rep["other_length_error"] = repr(e)[:200]
        rep["accepts_other_lengths"] = enc.varlen
    if not enc.lora:
        enc.ok_lora = False; rep["lora"] = enc.info.get("lora_problem", "no LoRA"); return rep
    try:
        xs = x10[:, :CROP]
        with torch.no_grad():
            LoRASwitch.on = False; z0 = enc.pool(enc.tokens(xs, amp=False), CROP)
            LoRASwitch.on = True; z1 = enc.pool(enc.tokens(xs, amp=False), CROP)
        scale = float(z0.abs().mean()) + 1e-12
        assert float((z1 - z0).abs().max()) / scale < 1e-4, "LoRA at zero changed the output"
        head_t = Head(enc.dim, CFG["emb_dim"]).to(DEVICE)
        arc_t = SubCenterArcFace(CFG["emb_dim"], 8, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
        opt_t = torch.optim.AdamW(lora_params_of(enc) + list(head_t.parameters()) + list(arc_t.parameters()), lr=1e-3)
        enc.train(); head_t.train(); arc_t.train(); times = []
        for k in range(3):
            xb = torch.from_numpy(rng0.uniform(-0.1, 0.1, (CFG["batch"], CROP)).astype(np.float32)).to(DEVICE)
            yb = torch.from_numpy(rng0.integers(0, 8, CFG["batch"])).to(DEVICE)
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
            loss = arc_t.loss(arc_t.cosine(head_t(enc.pool(enc.tokens(xb, amp=AMP), CROP))), yb)
            opt_t.zero_grad(set_to_none=True); loss.backward()
            if k == 0:
                assert math.isfinite(float(loss.detach())) and sum(float(w.B.grad.abs().sum()) for w in enc.lora if w.B.grad is not None) > 0, "no gradient reaches LoRA"
            opt_t.step()
            if DEVICE == "cuda":
                torch.cuda.synchronize()
            times.append(time.time() - t0)
        enc.eval()
        with torch.no_grad():
            moved = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = False
            off = float((enc.pool(enc.tokens(xs, amp=False), CROP) - z0).abs().max()) / scale
            LoRASwitch.on = True
        assert moved > 1e-6 and off < 1e-4, f"the LoRA switch does not work (moved {moved:.1e}, off {off:.1e})"
        enc.step_s = min(times[1:]) if timed else None
        rep["lora"] = enc.info.get("lora_on", "LoRA"); rep["step_seconds"] = round(enc.step_s, 4) if enc.step_s else None
        del head_t, arc_t, opt_t
    except Exception as e:
        enc.ok_lora = False; rep["lora"] = f"failed: {repr(e)[:300]}"
    finally:
        LoRASwitch.on = True; enc.reset_lora(); enc.eval()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return rep


def gate_gb(enc):
    # GB: extra energy in the top (bottom) 16 mel bins over the middle third, against the same input without it; the change of
    # each token, averaged per frequency row under each candidate layout
    with torch.no_grad():
        base = enc.wave2fb(torch.from_numpy(np.random.default_rng(5).normal(0, 0.003, (1, 10 * SR)).astype(np.float32)).to(DEVICE))
        a, b = base.shape[1] // 3, 2 * base.shape[1] // 3
        top, bot = base.clone(), base.clone()
        top[:, a:b, 112:128] += 2.0; bot[:, a:b, 0:16] += 2.0
        LoRASwitch.on = False
        try:
            h0, ht, hb = (enc.fb2tok(z, amp=False) for z in (base, top, bot))
        finally:
            LoRASwitch.on = True
    Lt = h0.shape[1]; Tp = Lt // 8
    dt = (ht - h0).norm(dim=-1)[0].cpu().numpy(); db = (hb - h0).norm(dim=-1)[0].cpu().numpy()
    out = {}
    for lay, row in (("frequency fastest", np.arange(Lt) % 8), ("time fastest", np.arange(Lt) // Tp)):
        pt = np.array([dt[row == f].mean() for f in range(8)]); pb = np.array([db[row == f].mean() for f in range(8)])
        it, ib = int(pt.argmax()), int(pb.argmax())
        rt = float(pt[it] / max(np.delete(pt, it).mean(), 1e-12)); rb = float(pb[ib] / max(np.delete(pb, ib).mean(), 1e-12))
        out[lay] = {"top_row": it, "bottom_row": ib, "ratio_top": round(rt, 2), "ratio_bottom": round(rb, 2),
                    "confirmed": bool({it, ib} == {0, 7} and min(rt, rb) >= GB_RATIO)}   # the two ends of the spectrum, one each
    if out["time fastest"]["confirmed"] and not out["frequency fastest"]["confirmed"]:
        enc.layout = "time fastest"
    out["used"] = enc.layout
    return out


ENC, FAILED, GBR = {}, {}, {}
ENC["BEATs"] = BeatsEnc("BEATs", model, LORA, {"checkpoint_sha256": CKPT_SHA})        # cell 9's model and LoRA blocks
ENC["BEATs"].step_s = STEP_S
with torch.no_grad():
    _gap = float((ENC["BEATs"].tokens(x4, amp=False) - tokens(x4, amp=False)).abs().max())
    _pool = float((ENC["BEATs"].pool(tokens(x4, amp=False), x4.shape[1]) - tokens(x4, amp=False).mean(1)).abs().max())
print(f"BEATs through the encoder wrapper against cell 9's forward pass: {_gap:.1e} (pooling {_pool:.1e})")
assert _gap < 1e-6 and _pool < 1e-5, "stop: the encoder wrapper does not reproduce cell 9"
for key in NEW_ENCODERS:
    t0 = time.time()
    try:
        enc = load_beats_iter3() if key == "BEATs-iter3" else load_eat(key)
        enc.info.update(smoke(enc))
        ENC[key] = enc
        print(f"{key}: ready · {json.dumps(enc.info)} · {time.time() - t0:.0f} s", flush=True)
    except Exception as e:
        FAILED[key] = repr(e)[:500]
        print(f"{key}: DROPPED, our loader failed ({FAILED[key]})", flush=True)
    gc.collect()
for key, enc in ENC.items():
    GBR[key] = gate_gb(enc)
    print(f"GB · {key}: {json.dumps(GBR[key])}")

In [ ]:
# 10 · Scorers (LDN1, B1, and BEAM's LDN1 per band), notebook 11's splits with the fold rule, notebook 20's frozen references, gate G0(b)
def unit(E):
    return E / np.linalg.norm(E, axis=-1, keepdims=True)


def b1(R, X):
    return 1.0 - (unit(X) @ unit(R).T).max(1)


def ldn1(R, X):
    # Wilkinghoff et al. (TASLP 2026), ratio form with K = 1: min over references y of d(x, y) / d(y, y₁)
    U = unit(R); Srr = U @ U.T; np.fill_diagonal(Srr, -np.inf)
    d1 = np.maximum(1.0 - Srr.max(1), LDN_FLOOR)
    return ((1.0 - unit(X) @ U.T) / d1[None, :]).min(1)


SCORE = {"B1": b1, "LDN1": ldn1}


def fold_of(p):
    return 1 if p in FOLDS[1] else 2


def fixed(E):
    return lambda p, s: E


def folded(E1, E2):
    # Split A: the model (or denoiser) of the machine's own fold; B and B+n: the other fold's, which never heard the machine
    return lambda p, s: (E1 if fold_of(p) == 1 else E2) if s == "A" else (E2 if fold_of(p) == 1 else E1)


def getter_of(E):
    return folded(*E) if isinstance(E, tuple) else fixed(E)


META, FZ = {}, {}
for t in TYPES:
    D = np.load(N20 / f"emb_{t}.npz", allow_pickle=False); Dn = np.load(N20 / f"emb_n2n_{t}.npz", allow_pickle=False)
    META[t] = (D["fname"], D["mid"], D["label"])
    for enc in ("BEATs", "ECHO"):
        FZ[(enc, t, "CH0")] = D[f"{enc}__CH0"].astype(np.float64)
        FZ[(enc, t, "MWF")] = D[f"{enc}__MWF"].astype(np.float64)
        FZ[(enc, t, "N2N-CH0")] = tuple(Dn[f"{enc}__N2N-CH0__{k}"].astype(np.float64) for k in (1, 2))
SPLIT = {t: {p: split_a(META[t][1], META[t][2], META[t][0], p) for p in sorted(set(META[t][1].tolist()))} for t in TYPES}
IS_TRAIN = {}
for t in TYPES:
    IS_TRAIN[t] = np.zeros(len(META[t][0]), bool)
    for p in SPLIT[t]:
        IS_TRAIN[t][SPLIT[t][p][0]] = True


def evaluate(get, t, scorer=ldn1):
    # notebook 11's splits; get(p, split) → the embedding matrix for machine p of type t
    ids = sorted(SPLIT[t]); label = META[t][2]
    r = {s: {} for s in SPLITS}
    for p in ids:
        tr, te = SPLIT[t][p]; y = label[te]
        fleet = np.concatenate([SPLIT[t][q][0] for q in ids if q != p])
        for s, ref in (("A", tr), ("B", fleet), *[(f"B+{n}", np.r_[fleet, tr[:n]]) for n in SHOTS]):
            E = get(p, s)
            r[s][p] = round(float(roc_auc_score(y, scorer(E[ref], E[te]))), 4)
    for s in SPLITS:
        r[s]["avg"] = round(float(np.mean([r[s][p] for p in ids])), 4)
    return r


def S_of(res, types):
    # the rules' score: mean over the types' machines of (A + B+10)/2 (each type has four machines)
    return round(float(np.mean([(res[t]["A"]["avg"] + res[t]["B+10"]["avg"]) / 2 for t in types])), 4)


N20RES = json.loads((N20 / "denoise_score.json").read_text())
FROZEN, worst = {}, 0.0
for tr_, inp in TRACK_INPUT.items():
    for sc in ("LDN1", "B1"):
        FROZEN[(tr_, sc)] = {t: evaluate(getter_of(FZ[("BEATs", t, inp)]), t, SCORE[sc]) for t in TYPES}
        for t in TYPES:
            ref = N20RES["results"][f"BEATs|{t}|{inp}|{sc}"]
            worst = max(worst, max(abs(FROZEN[(tr_, sc)][t][s][p] - ref[s][p]) for s in SPLITS for p in ref[s]))
FROZEN_S = {tr_: (S_of(FROZEN[(tr_, "LDN1")], MAIN), S_of(FROZEN[(tr_, "LDN1")], AUX)) for tr_ in TRACK_INPUT}
ref_gap = max(abs(FROZEN_S[tr_][i] - FROZEN_REF[tr_][i]) for tr_ in TRACK_INPUT for i in (0, 1))
G0 = {"a": {}, "b": {"worst_machine": round(worst, 4), "frozen_S": FROZEN_S, "gap_to_rules": round(ref_gap, 4)}, "c": {}}
print(f"G0(b) · notebook 20's frozen results recomputed: largest |ΔAUC| {worst:.4f} (tolerance {G0B_TOL})")
print("frozen references (development, confirmation):", FROZEN_S, f"· largest gap to the rules' numbers {ref_gap:.4f}")
if not TEST and (worst > G0B_TOL or ref_gap > G0B_TOL):
    raise SystemExit("stop: G0(b) failed, notebook 20's files do not give its numbers")


def evaluate_fn(fn, t):
    # notebook 11's splits; fn(p, split, reference rows, test rows) → anomaly scores of the test rows
    ids = sorted(SPLIT[t]); label = META[t][2]
    r = {s: {} for s in SPLITS}
    for p in ids:
        tr, te = SPLIT[t][p]; y = label[te]
        fleet = np.concatenate([SPLIT[t][q][0] for q in ids if q != p])
        for s, ref in (("A", tr), ("B", fleet), *[(f"B+{n}", np.r_[fleet, tr[:n]]) for n in SHOTS]):
            r[s][p] = round(float(roc_auc_score(y, fn(p, s, ref, te))), 4)
    for s in SPLITS:
        r[s]["avg"] = round(float(np.mean([r[s][p] for p in ids])), 4)
    return r


def band_ldn1(R, X):
    # BEAM with LDN1 per band (K = 1, cosine distance): R (references, bands, D) and X (tests, bands, D), unit per band,
    # float32 on the device → the mean over bands of min over references y of d(x, y) / d(y, y₁)
    Rb, Xb = R.transpose(0, 1), X.transpose(0, 1)
    Srr = torch.bmm(Rb, Rb.transpose(1, 2))
    Srr.diagonal(dim1=1, dim2=2).fill_(-float("inf"))
    d1 = (1.0 - Srr.max(2).values).clamp_min(LDN_FLOOR)
    return ((1.0 - torch.bmm(Xb, Rb.transpose(1, 2))) / d1[:, None, :]).min(2).values.mean(0)


# band LDN1 equals global LDN1 when there is one band (a check of the algebra)
_R = np.random.default_rng(3).standard_normal((40, 12)); _X = np.random.default_rng(4).standard_normal((9, 12))
_b = band_ldn1(F.normalize(torch.from_numpy(_R).float().to(DEVICE), dim=-1)[:, None], F.normalize(torch.from_numpy(_X).float().to(DEVICE), dim=-1)[:, None])
assert np.allclose(_b.cpu().numpy(), ldn1(_R, _X), atol=1e-4), "stop: band LDN1 does not reduce to LDN1"

In [ ]:
# 11 · The audio: notebook 20's streams, its denoisers on microphone 0, the DCASE 2020 extra machines; gates G0(a), G0(c)
ST, XS, XMETA, XFZ, NETS, PDW, GT, XLEVEL, PREP_S, SAT = {}, {}, {}, {}, {}, {}, {}, {}, {}, {}
TRAIN_ROWS = {(t, p): SPLIT[t][p][0] for t in TYPES for p in SPLIT[t]}


def to_int16(v):
    # only for microphone-0 samples read from the 16-bit WAV files, so this conversion is exact
    return np.clip(np.round(v * 32768.0), -32768, 32767).astype(np.int16)


def f32(a):
    # a slice of a stream → float32 signal: int16 streams hold samples × 32768, float16 streams hold the signal itself
    a = np.asarray(a)
    return a.astype(np.float32) / 32768.0 if a.dtype == np.int16 else a.astype(np.float32)


_v = np.array([1.7, -2.3, 0.25], np.float32)
assert np.array_equal(f32(_v.astype(np.float16)), _v.astype(np.float16).astype(np.float32)) and abs(f32(np.int16([16384]))[0] - 0.5) < 1e-9


def mwf_weights(t):
    D = np.load(N19 / f"stats_{t}.npz", allow_pickle=False)
    return {k.split("__")[0]: D[k] for k in D.files if k.endswith("__w__MWF|-6dB")}


def build_streams(t, zfs, names, mid, W):
    # notebook 20's four streams, one pass over every clip: microphone 0 as int16 (exact), MWF as float16 (no clipping)
    n = len(names); ns = read2(zfs, names[0])["6dB"].shape[0]
    mm = {s: np.lib.format.open_memmap(STORE / f"{t}_{s}.npy", mode="w+", dtype=np.float16 if s in FLOAT_STREAMS else np.int16, shape=(n, ns))
          for s in STREAMS}
    Wt = {p: torch.from_numpy(w.astype(np.complex64)).to(DEVICE) for p, w in W.items()}
    peaks = {s: np.zeros(n, np.float32) for s in FLOAT_STREAMS}
    t0 = time.time()
    for i, x in enumerate(stream(lambda i: read2(zfs, names[i]), range(n))):
        p = mid[i]
        sig = {"CH0_m6": np.ascontiguousarray(x["-6dB"][:, 0]), "CH0_6": np.ascontiguousarray(x["6dB"][:, 0]),
               "MWF_m6": istft(beam(Wt[p], stft(to_dev(x["-6dB"]))), ns).cpu().numpy(),
               "MWF_6": istft(beam(Wt[p], stft(to_dev(x["6dB"]))), ns).cpu().numpy()}
        for s in STREAMS:
            if s in FLOAT_STREAMS:
                mm[s][i] = sig[s].astype(np.float16); peaks[s][i] = float(np.abs(sig[s]).max())
            else:
                mm[s][i] = to_int16(sig[s])
        if i % 1000 == 0:
            print(f"  {t}: {i}/{n} clips · {time.time() - t0:.0f} s", flush=True)
    for s in STREAMS:
        mm[s].flush()
    for s in FLOAT_STREAMS:
        SAT[f"{t}|{s}"] = peak_info(peaks[s], META[t][2])
    (STORE / f"{t}_peaks.json").write_text(json.dumps({s: SAT[f"{t}|{s}"] for s in FLOAT_STREAMS}))


def n2n_nets(t):
    # notebook 20's microphone-0 denoisers of this type, one per fold
    nets = {}
    for k in (1, 2):
        rec = torch.load(N20 / f"n2n_{t}_CH0_f{k}.pt", map_location="cpu", weights_only=False)
        assert rec["base"] == "CH0", rec["base"]
        net = N2NNet().to(DEVICE); net.load_state_dict(rec["state"]); net.eval(); nets[k] = net
    return nets


def peak_info(pk, labels=None):
    # how many clips reach |x| ≥ 1 (which 16-bit storage clips), overall and, for MIMII clips, by label
    out = {"clips_peak_at_least_1": round(float(np.mean(pk >= 1.0)), 4), "max_abs": round(float(pk.max()), 3)}
    if labels is not None:
        for nm, v in (("normal", 0), ("abnormal", 1)):
            out[nm] = round(float(np.mean(pk[labels == v] >= 1.0)), 4) if np.any(labels == v) else None
    return out


def apply_n2n(net, src_mm, path, batch=16, labels=None):
    # the denoiser on whole clips, written to a float16 stream (via a temporary file, so a crash leaves nothing half-written)
    tmp = path.with_name(path.stem + ".part.npy")
    n, ns = src_mm.shape
    mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.float16, shape=(n, ns))
    pk = np.zeros(n, np.float32)
    for i in range(0, n, batch):
        y = denoise(net, torch.from_numpy(f32(src_mm[i:i + batch])).to(DEVICE)).cpu().numpy()
        mm[i:i + batch] = y.astype(np.float16); pk[i:i + len(y)] = np.abs(y).max(1)
    mm.flush(); del mm
    os.replace(tmp, path)
    info = peak_info(pk, labels)
    path.with_name(path.stem + "_peaks.json").write_text(json.dumps(info))
    return info


@torch.no_grad()
def frozen_emb(mm, rows=None, batch=None):
    # frozen BEATs (LoRA switched off) on whole clips of a stream → clip embeddings (n, 768) float32
    batch = batch or CFG["embed_batch"]; rows = np.arange(mm.shape[0]) if rows is None else rows
    LoRASwitch.on = False; model.eval(); out = []
    try:
        for i in range(0, len(rows), batch):
            x = torch.from_numpy(f32(mm[rows[i:i + batch]])).to(DEVICE)
            out.append(tokens(x, amp=AMP).mean(1).cpu().numpy())
    finally:
        LoRASwitch.on = True
    return np.concatenate(out)


def build_xc(t):
    # DCASE 2020's extra machines of this type: the first CFG["xc_cap"] normal clips by name per machine, microphone 0 at −6 dB
    meta_f = STORE / f"xc_{t}.json"
    if meta_f.exists() and all((STORE / f"xc_{t}_{s}.npy").exists() for s in ("CH0", "N2N1", "N2N2")):
        return json.loads(meta_f.read_text())
    zf = zipfile.ZipFile(get_file(XZIPS[t]))
    names = np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav")))
    mid_all = np.array([re.search(r"(id_\d+)", n).group(1) for n in names])
    if not TEST:
        assert all(n.replace("\\", "/").split("/")[-1].startswith("normal") for n in names), "stop: the additional training data should be normal clips only"
    keep = np.concatenate([np.where(mid_all == q)[0][:CFG["xc_cap"]] for q in sorted(set(mid_all.tolist()))])
    names, mid = names[keep], mid_all[keep]
    assert not (set(mid.tolist()) & set(META[t][1].tolist())), "stop: an extra machine has a MIMII machine's ID"
    ns = read_all(zf.read(names[0]))[0].shape[0]
    mm = np.lib.format.open_memmap(STORE / f"xc_{t}_CH0.npy", mode="w+", dtype=np.int16, shape=(len(names), ns))
    for i, (x, sr) in enumerate(stream(lambda i: read_all(zf.read(names[i])), range(len(names)))):
        assert sr == SR and x.shape[0] == ns, (names[i], sr, x.shape)
        mm[i] = to_int16(x[:, 0])
    mm.flush(); del mm; zf.close()
    src_mm = np.load(STORE / f"xc_{t}_CH0.npy", mmap_mode="r")
    for k in (1, 2):
        SAT[f"{t}|XC_N2N{k}"] = apply_n2n(NETS[t][k], src_mm, STORE / f"xc_{t}_N2N{k}.npy")
    info = {"names": names.tolist(), "mid": mid.tolist(), "per_machine": dict(Counter(mid.tolist())), "samples": int(ns),
            "zip": XZIPS[t][1], "zip_md5": XZIPS[t][2]}
    meta_f.write_text(json.dumps(info))
    if IN_COLAB:
        (WORK / XZIPS[t][1]).unlink(missing_ok=True)
    return info


def twin_power(t, dom):
    # the mean power of the noise-only twin difference y₋₆ − y₆, for training clips only (a test clip's twin is never read)
    a, b = ST[t][f"{dom}_m6"], ST[t][f"{dom}_6"]
    P = np.full(a.shape[0], np.nan)
    rows = np.where(IS_TRAIN[t])[0]
    for i in range(0, len(rows), 64):
        r = rows[i:i + 64]
        P[r] = np.mean((f32(a[r]).astype(np.float64) - f32(b[r])) ** 2, 1)
    return P


def swap_full(t, dom, i, j, shift):
    # TS on whole clips: clip i's 6 dB twin plus the power-matched, circularly shifted noise of clip j (both training clips)
    assert IS_TRAIN[t][i] and IS_TRAIN[t][j] and META[t][1][i] == META[t][1][j]
    st = ST[t]; g = math.sqrt(PDW[(t, dom)][i] / PDW[(t, dom)][j])
    D = f32(st[f"{dom}_m6"][j]) - f32(st[f"{dom}_6"][j])
    return f32(st[f"{dom}_6"][i]) + C_SWAP * g * np.roll(D, shift)


def gate_gt(t):
    rng = np.random.default_rng(0)
    for dom in ("MWF", "CH0"):
        ratios = []
        for _ in range(GT_PAIRS):
            p = sorted(SPLIT[t])[int(rng.integers(0, len(SPLIT[t])))]
            i, j = rng.choice(TRAIN_ROWS[(t, p)], 2, replace=False)
            y = swap_full(t, dom, int(i), int(j), int(rng.integers(0, ST[t][f"{dom}_m6"].shape[1])))
            ratios.append(float(np.mean(y.astype(np.float64) ** 2) / np.mean(f32(ST[t][f"{dom}_m6"][i]).astype(np.float64) ** 2)))
        GT[(t, dom)] = round(float(np.median(ratios)), 4)
    print(f"GT · {t}: median power of the swap over the −6 dB clip: MWF {GT[(t, 'MWF')]}, microphone 0 {GT[(t, 'CH0')]} (range {GT_RANGE})")


def gate_g0c(t):
    m = min(G0C_CLIPS, len(META[t][0])); rows = np.arange(m)
    pairs = {"CH0": (ST[t]["CH0_m6"], FZ[("BEATs", t, "CH0")]), "MWF": (ST[t]["MWF_m6"], FZ[("BEATs", t, "MWF")]),
             "N2N-CH0 fold 1": (ST[t]["N2N1_m6"], FZ[("BEATs", t, "N2N-CH0")][0]),
             "N2N-CH0 fold 2": (ST[t]["N2N2_m6"], FZ[("BEATs", t, "N2N-CH0")][1])}
    t0, cos, worst = time.time(), {}, {}
    for k, (mm, ref) in pairs.items():
        e = frozen_emb(mm, rows)
        c = np.sum(unit(e.astype(np.float64)) * unit(ref[:m]), 1); w = int(np.argmin(c))
        cos[k] = round(float(c[w]), 6)
        worst[k] = {"clip": str(META[t][0][w]), "cosine": round(float(c[w]), 6), "peak_abs": round(float(np.abs(f32(mm[w])).max()), 3)}
    G0["c"][t] = cos; G0.setdefault("c_worst", {})[t] = worst
    PREP_S["embed_clips_per_s"] = round(4 * m / (time.time() - t0), 1)
    print(f"G0(c) · {t}: frozen BEATs on our streams against notebook 20's embeddings, smallest cosine {cos} (bar {G0C_COS})")
    if not TEST and min(cos.values()) < G0C_COS:
        print("the worst clip of each stream:", json.dumps(worst, indent=1))
        raise SystemExit("stop: G0(c) failed, the streams or the denoisers are not notebook 20's")


def prepare(t, nxt=None):
    t0 = time.time(); names, mid, label = META[t]
    NETS[t] = n2n_nets(t)
    if not (STORE / f"{t}.json").exists():
        W = mwf_weights(t)
        assert set(W) == set(mid.tolist()), f"{t}: MWF weights for {sorted(W)} but machines {sorted(set(mid.tolist()))}"
        if t in PREFETCH:
            PREFETCH[t].join()
        zfs = open2(t)
        ok = all(np.array_equal(np.array(sorted(n for n in zf.namelist() if n.lower().endswith(".wav"))), names) for zf in zfs.values())
        G0["a"][t] = bool(ok)
        print(f"G0(a) · {t}: the clips are notebook 20's: {ok}")
        if not ok and not TEST:
            raise SystemExit("stop: G0(a) failed")
        build_streams(t, zfs, names, mid, W)
        for zf in zfs.values():
            zf.close()
        if IN_COLAB:
            for lv in LEVELS:
                (WORK / ZIPS[f"{t}|{lv}"][1]).unlink(missing_ok=True)
        (STORE / f"{t}.json").write_text(json.dumps({"n": int(len(names))}))
    else:
        G0["a"].setdefault(t, "local store from an earlier run of this notebook in this session")
        SAT.update({f"{t}|{s}": v for s, v in json.loads((STORE / f"{t}_peaks.json").read_text()).items()})
    prefetch(nxt)
    ST[t] = {s: np.load(STORE / f"{t}_{s}.npy", mmap_mode="r") for s in STREAMS}
    for k in (1, 2):
        pth = STORE / f"{t}_N2N{k}_m6.npy"
        if not pth.exists():
            apply_n2n(NETS[t][k], ST[t]["CH0_m6"], pth, labels=label)
        SAT[f"{t}|N2N{k}_m6"] = json.loads(pth.with_name(pth.stem + "_peaks.json").read_text())
        ST[t][f"N2N{k}_m6"] = np.load(pth, mmap_mode="r")
    XMETA[t] = build_xc(t)
    XS[t] = {s: np.load(STORE / f"xc_{t}_{s}.npy", mmap_mode="r") for s in ("CH0", "N2N1", "N2N2")}
    gate_g0c(t)
    rms = lambda mm, rows: float(np.median(np.sqrt(np.mean(f32(mm[rows]).astype(np.float64) ** 2, 1))))
    tr_rows = np.where(IS_TRAIN[t])[0][:200]
    XLEVEL[t] = round(rms(XS[t]["CH0"], np.arange(min(200, len(XMETA[t]["mid"])))) / rms(ST[t]["CH0_m6"], tr_rows), 3)
    PREP_S[t] = round(time.time() - t0, 1)
    print(f"{t}: ready · {len(names)} MIMII clips · extra machines {XMETA[t]['per_machine']} (RMS ratio to MIMII −6 dB {XLEVEL[t]})"
          f" · {PREP_S[t]:.0f} s", flush=True)
    print("  share of clips whose processed signal peaks at |x| ≥ 1 (16-bit storage would have clipped them):",
          {k.split("|")[1]: v["clips_peak_at_least_1"] for k, v in SAT.items() if k.startswith(t + "|")})


for j_, t in enumerate(TYPES):
    prepare(t, nxt=TYPES[j_ + 1] if j_ + 1 < len(TYPES) else None)
print(f"all data ready after {(time.time() - T_START) / 60:.0f} min")
_spe = {tr: (sum(len(TRAIN_ROWS[k]) for k in TRAIN_ROWS) / 2 + (sum(len(XMETA[t]["mid"]) for t in TYPES) if "XC" in RECIPE[tr][0] else 0)) / CFG["batch"]
        for tr in TRACKS}
for k_, e_ in ENC.items():
    if k_ != "BEATs" and e_.ok_lora:
        _h = sum(len(RECIPE[tr][1]) * 2 * _spe[tr] * CFG["epochs"] * e_.step_s * 1.15 for tr in TRACKS) / 3600
        print(f"  if {k_} is selected: its {sum(2 * len(RECIPE[tr][1]) for tr in TRACKS)} trainings take about {_h:.1f} h on this GPU")

In [ ]:
# 12 · Training any encoder with a track's recipe; passes over every clip (global and band views); systems and their scores
ENCT = {"BEATs": "B", "BEATs-iter3": "B3", "EAT-base": "EB", "EAT-large": "EL"}
TRK = {"ARR": "ARR", "MIC+": "MICp", "MIC": "MIC"}
ECHO_INPUT = {"CH0": "CH0", "MWF": "MWF", "N2N-CH0": "N2N-CH0"}
VIEW_DIR = WORK / "views22"; VIEW_DIR.mkdir(exist_ok=True)
TRACK_IN = dict(TRACK_INPUT)


def n21_tag(track, flags, seed, fold):
    return f"{TRK[track]}_{'-'.join(f for f in ('XC', 'PL', 'TS', 'PD') if f in flags) or 'R0'}_s{seed}_f{fold}"


def n22_tag(ek, track, inp, flags, seed, fold):
    return f"{ENCT[ek]}_{TRK[track]}{'n' if inp == 'N2N-MWF' else ''}_{'-'.join(flags) or 'R0'}_s{seed}_f{fold}"


def stream_name(inp, fold):
    # the stream a model of fold k reads: denoised inputs use denoiser k, so no learned part has heard a new machine
    return {"MWF": "MWF_m6", "CH0": "CH0_m6", "N2N-CH0": f"N2N{fold}_m6", "N2N-MWF": f"N2NMWF{fold}_m6"}[inp]


def training_set(track, flags, fold):
    # items (type index, source 0 = MIMII / 1 = XC, row, class): one class per machine of the fold, and per extra machine with XC
    items, classes = [], []
    for ti, t in enumerate(TYPES):
        for p in FOLDS[fold]:
            if p in SPLIT[t]:
                c = len(classes); classes.append(f"{t}|{p}"); items.extend((ti, 0, int(r), c) for r in TRAIN_ROWS[(t, p)])
    if "XC" in flags:
        for ti, t in enumerate(TYPES):
            xm = np.array(XMETA[t]["mid"])
            for q in sorted(set(xm.tolist())):
                c = len(classes); classes.append(f"{t}|{q}|DCASE2020"); items.extend((ti, 1, int(r), c) for r in np.where(xm == q)[0])
    return np.array(items, dtype=np.int64), classes


def win(mm, row, s, n):
    return f32(mm[row, s:s + n])


def make_batch(track, inp, fold, items, ids, rng, n, want_teacher):
    # the student's crops; PD's teacher crops: the 6 dB twin of the same window through MWF (ARR only)
    B = len(ids); x = np.zeros((B, n), np.float32); tch = np.zeros((B, n), np.float32) if want_teacher else None
    has = np.zeros(B, bool)
    for b, it in enumerate(ids):
        ti, source, r, _ = items[it]; t = TYPES[ti]
        if source == 1:
            mm = XS[t]["CH0" if track == "MIC" else f"N2N{fold}"]
            x[b] = win(mm, r, int(rng.integers(0, mm.shape[1] - n + 1)), n)
            continue
        mm = ST[t][stream_name(inp, fold)]; s = int(rng.integers(0, mm.shape[1] - n + 1))
        x[b] = win(mm, r, s, n)
        if want_teacher:
            assert track == "ARR" and IS_TRAIN[t][r]
            tch[b] = win(ST[t]["MWF_6"], r, s, n); has[b] = True
    T = torch.from_numpy(tch).to(DEVICE) if want_teacher else None
    return torch.from_numpy(x).to(DEVICE), T, torch.from_numpy(has).to(DEVICE)


def fresh(enc, seed, n_classes):
    # LoRA back to zero, a new head and a new ArcFace layer: every training starts from the same frozen encoder
    torch.manual_seed(seed); enc.reset_lora()
    head = Head(enc.dim, CFG["emb_dim"]).to(DEVICE)
    arc = SubCenterArcFace(CFG["emb_dim"], n_classes, CFG["subcenters"], CFG["arc_scale"], CFG["arc_margin"]).to(DEVICE)
    return head, arc


def adapter_state(enc, head, arc):
    return {"lora": [{"A": w.A.detach().cpu(), "B": w.B.detach().cpu()} for w in enc.lora],
            "head": {k: v.cpu() for k, v in head.state_dict().items()}, "arc": {k: v.cpu() for k, v in arc.state_dict().items()}}


def load_lora(enc, lst):
    assert len(lst) == len(enc.lora), (len(lst), len(enc.lora))
    with torch.no_grad():
        for w, s in zip(enc.lora, lst):
            w.A.copy_(s["A"].to(DEVICE)); w.B.copy_(s["B"].to(DEVICE))


def teacher_tokens(enc, T):
    # the frozen encoder (LoRA off, evaluation mode, no gradient) on the teacher crops
    with torch.no_grad():
        LoRASwitch.on = False; enc.eval()
        try:
            return enc.tokens(T, amp=AMP)
        finally:
            LoRASwitch.on = True; enc.train()


def step_losses(enc, arc, head, X, y, mix, lam, perm, T, has, n):
    tok = enc.tokens(X, amp=AMP)
    cos = arc.cosine(head(enc.pool(tok, n)))
    ce = lam * arc.loss(cos, y) + (1.0 - lam) * arc.loss(cos, y[perm]) if mix else arc.loss(cos, y)
    pd = None
    if T is not None and bool(has.any()):
        tt = teacher_tokens(enc, T[has])
        pd = (1.0 - F.cosine_similarity(tok[has], tt, dim=-1)).mean()
    return ce, pd


def train(ek, track, inp, flags, seed, fold):
    # notebook 21's training for any encoder: LoRA + head + sub-centre ArcFace, mixup, warm-up and cosine decay, PD's λ rule
    enc = ENC[ek]; tag = n22_tag(ek, track, inp, flags, seed, fold)
    final, resume = OUT / f"ft_{tag}.pt", OUT / f"ft_{tag}_resume.pt"
    items, classes = training_set(track, flags, fold)
    head, arc = fresh(enc, seed, len(classes))
    if final.exists():
        st = torch.load(final, map_location="cpu", weights_only=False)
        assert st["classes"] == classes, "the saved model used other classes"
        load_lora(enc, st["lora"]); head.load_state_dict(st["head"]); arc.load_state_dict(st["arc"])
        return head, st["log"]
    pd_on = "PD" in flags
    params = lora_params_of(enc) + list(head.parameters()) + list(arc.parameters())
    opt = torch.optim.AdamW([{"params": lora_params_of(enc), "lr": CFG["lr_lora"], "weight_decay": 0.0},
                             {"params": list(head.parameters()) + list(arc.parameters()), "lr": CFG["lr_head"], "weight_decay": CFG["wd_head"]}])
    spe = len(items) // CFG["batch"]
    total, warm = spe * CFG["epochs"], spe * CFG["warmup_epochs"]
    lr_f = lambda s: (s + 1) / warm if s < warm else 0.5 * (1.0 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, [lr_f, lr_f])
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    log = {"tag": tag, "encoder": ek, "track": track, "input": inp, "flags": list(flags), "seed": seed, "fold": fold, "cfg": CFG,
           "classes": classes, "n_train": int(len(items)), "n_xc": int((items[:, 1] == 1).sum()), "lambda_pd": None,
           "epochs": [], "resumed_at_epoch": []}
    start, n = 0, CROP
    if resume.exists():
        st = torch.load(resume, map_location="cpu", weights_only=False)
        load_lora(enc, st["lora"]); head.load_state_dict(st["head"]); arc.load_state_dict(st["arc"])
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        log, start = st["log"], st["epoch"] + 1
        log["resumed_at_epoch"].append(start); print(f"resuming {tag} at epoch {start}")
    if pd_on and log["lambda_pd"] is None:
        # the rules' balance: λ_PD = mean ArcFace loss / mean distillation loss over a few batches, before any update
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), 999, ENC_ORDER.index(ek)]); ce_l, pd_l = [], []
        enc.train(); head.train(); arc.train()
        with torch.no_grad():
            for _ in range(CFG["pd_balance_batches"]):
                ids = rng.choice(len(items), CFG["batch"], replace=False)
                X, T, has = make_batch(track, inp, fold, items, ids, rng, n, True)
                y = torch.from_numpy(items[ids, 3]).to(DEVICE)
                ce, pd = step_losses(enc, arc, head, X, y, False, 1.0, None, T, has, n)
                ce_l.append(float(ce)); pd_l.append(float(pd)) if pd is not None else None
        log["lambda_pd"] = round(float(np.mean(ce_l) / max(np.mean(pd_l), 1e-6)), 4)
        print(f"{tag}: λ_PD = {log['lambda_pd']} (ArcFace {np.mean(ce_l):.3f}, distillation {np.mean(pd_l):.4f} at the start)")
    print(f"{tag}: {len(items)} training clips ({log['n_xc']} from DCASE 2020) · {len(classes)} classes · {spe} steps per epoch", flush=True)
    for ep in range(start, CFG["epochs"]):
        rng = np.random.default_rng([seed, fold, ALL_TRACKS.index(track), ep, ENC_ORDER.index(ek)])
        np.random.seed(seed * 1000 + ep)                              # BEATs draws its layer-drop from numpy's global stream
        order = rng.permutation(len(items))
        enc.train(); head.train(); arc.train()
        t0, tot, tce, tpd, n_mix = time.time(), 0.0, 0.0, 0.0, 0
        for b in range(spe):
            ids = order[b * CFG["batch"]:(b + 1) * CFG["batch"]]
            mix = bool(rng.random() < CFG["mixup_prob"])
            X, T, has = make_batch(track, inp, fold, items, ids, rng, n, pd_on)
            y = torch.from_numpy(items[ids, 3]).to(DEVICE)
            lam, perm = 1.0, None
            if mix:
                lam = float(rng.beta(CFG["mixup_beta"], CFG["mixup_beta"]))
                perm = torch.from_numpy(rng.permutation(len(ids))).to(DEVICE)
                X = lam * X + (1.0 - lam) * X[perm]
                if pd_on:
                    T = lam * T + (1.0 - lam) * T[perm]; has = has & has[perm]
            ce, pd = step_losses(enc, arc, head, X, y, mix, lam, perm, T, has, n)
            loss = ce if pd is None else ce + log["lambda_pd"] * pd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, CFG["clip_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss.detach()); tce += float(ce.detach()); tpd += 0.0 if pd is None else float(pd.detach()); n_mix += mix
        log["epochs"].append({"epoch": ep, "mean_loss": round(tot / spe, 4), "mean_arcface": round(tce / spe, 4),
                              "mean_distillation": round(tpd / spe, 5), "mixup_batches": int(n_mix), "seconds": round(time.time() - t0, 1)})
        print(f"{tag} · epoch {ep + 1}/{CFG['epochs']} · loss {tot / spe:.3f}" + (f" (distillation {tpd / spe:.4f})" if pd_on else "")
              + f" · {time.time() - t0:.0f} s", flush=True)
        torch.save({**adapter_state(enc, head, arc), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "log": log}, resume)
    enc.eval(); head.eval(); arc.eval()
    torch.save({**adapter_state(enc, head, arc), "classes": classes, "log": log}, final)
    resume.unlink(missing_ok=True)
    return head, log


@torch.no_grad()
def run_pass(enc, mm, head=None, rows=None, views=True):
    # whole clips through the encoder as it stands (LoRA as switched): per clip the global mean P, F = head(P) when a head is
    # given, and the band views: per frequency row, the mean, the max and RDP (γ = 4) over the time patches
    rows = np.arange(mm.shape[0]) if rows is None else rows
    n, bs = mm.shape[1], CFG["embed_batch"]; enc.eval()
    P = np.zeros((len(rows), enc.dim), np.float16); Fh = np.zeros((len(rows), CFG["emb_dim"]), np.float16) if head is not None else None
    V = {v: np.zeros((len(rows), 8, enc.dim), np.float16) for v in VIEWS} if views else None
    for i in range(0, len(rows), bs):
        x = torch.from_numpy(f32(mm[rows[i:i + bs]])).to(DEVICE); m_ = len(x)
        g = enc.grid(enc.tokens(x, amp=AMP), n)
        p = g.mean((1, 2)); P[i:i + m_] = p.cpu().numpy()
        if head is not None:
            Fh[i:i + m_] = head(p).cpu().numpy()
        if views:
            V["mean"][i:i + m_] = g.mean(1).cpu().numpy(); V["max"][i:i + m_] = g.amax(1).cpu().numpy()
            d = (g - g.mean(1, keepdim=True)).norm(dim=-1)            # RDP (arXiv 2603.04605): deviation from the band's mean
            dh = d / d.amax(1, keepdim=True).clamp_min(1e-12)
            w = (1.0 + dh) ** RDP_GAMMA; w = w / w.sum(1, keepdim=True)
            V[f"rdp{RDP_GAMMA}"][i:i + m_] = (w.unsqueeze(-1) * g).sum(1).cpu().numpy()
    return P, Fh, V


PASS, T_PASS = {}, {}


def model_pass(ek, ft_path, inp, fold):
    # one model (the encoder, frozen when ft_path is None, else with the adapters and head of ft_path) on the input of fold k:
    # every clip, every view; kept on the local disk and recomputed after a restart
    sname = stream_name(inp, fold)
    key = f"{ek}__{ft_path.stem if ft_path is not None else 'frozen'}__{sname}"
    d = VIEW_DIR / key
    if not (d / "done.json").exists():
        enc = ENC[ek]; head = None; t0 = time.time()
        if ft_path is not None:
            st = torch.load(ft_path, map_location="cpu", weights_only=False)
            load_lora(enc, st["lora"])
            head = Head(enc.dim, CFG["emb_dim"]).to(DEVICE); head.load_state_dict(st["head"]); head.eval()
            LoRASwitch.on = True
        else:
            LoRASwitch.on = False
        d.mkdir(parents=True, exist_ok=True)
        try:
            for t in TYPES:
                P, Fh, V = run_pass(enc, ST[t][sname], head)
                np.save(d / f"{t}_P.npy", P)
                if Fh is not None:
                    np.save(d / f"{t}_F.npy", Fh)
                for v, a in V.items():
                    np.save(d / f"{t}_{v}.npy", a)
        finally:
            LoRASwitch.on = True
        (d / "done.json").write_text(json.dumps({"F": head is not None}))
        T_PASS[key] = round(time.time() - t0, 1)
        print(f"  pass {key}: {T_PASS[key]:.0f} s", flush=True)
    if key not in PASS:
        hasf = json.loads((d / "done.json").read_text())["F"]
        PASS[key] = {t: {nm: np.load(d / f"{t}_{nm}.npy", mmap_mode="r") for nm in ["P"] + (["F"] if hasf else []) + VIEWS} for t in TYPES}
    return PASS[key]


LOGS, T_TRAIN = {}, {}


def ensure(ek, track, inp, flags, seed):
    # both folds of one recipe and seed: trained (or resumed, or read from Drive), passed over every clip; returns gate G1
    ok = True
    for fold in (1, 2):
        tag = n22_tag(ek, track, inp, flags, seed, fold); ef, fin = OUT / f"emb_{tag}.npz", OUT / f"ft_{tag}.pt"
        if not (ef.exists() and fin.exists()):
            t0 = time.time()
            _, log = train(ek, track, inp, flags, seed, fold)
            res = model_pass(ek, fin, inp, fold)
            tmp = OUT / f"emb_{tag}.part.npz"
            np.savez(tmp, **{f"{nm}__{t}": np.asarray(res[t][nm]) for t in TYPES for nm in ("P", "F")},
                     meta=json.dumps({"tag": tag, "weights_md5": md5(fin), "encoder": ek, "test_run": TEST, "date": time.strftime("%Y-%m-%d %H:%M")}))
            os.replace(tmp, ef)
            T_TRAIN[tag] = round(time.time() - t0, 1)
            print(f"saved {ef.name} · {T_TRAIN[tag] / 60:.1f} min for training and the pass", flush=True)
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        log = torch.load(fin, map_location="cpu", weights_only=False)["log"]
        LOGS[tag] = log
        ok = ok and log["epochs"][-1]["mean_loss"] < log["epochs"][0]["mean_loss"]
    return ok


# ---- systems: {"enc", "inp", "scorer", "kind", "ef", "ft"} or a fusion {"parts": [system, system]} ----
def sysd(ek, inp, scorer="global", kind="P", ef=False, ft=None):
    return {"enc": ek, "inp": inp, "scorer": scorer, "kind": kind, "ef": ef, "ft": ft}


def ft_models(sd, fold):
    # the adapter files of a fine-tuned system for fold k, one per seed
    f = sd["ft"]
    if f["src"] == "N21":
        return [N21 / f"ft_{n21_tag(f['track'], f['flags'], s, fold)}.pt" for s in f["seeds"]]
    return [OUT / f"ft_{n22_tag(sd['enc'], f['track'], sd['inp'], f['flags'], s, fold)}.pt" for s in f["seeds"]]


def echo_of(t, inp, fold):
    e = FZ[("ECHO", t, ECHO_INPUT[inp])]
    return e[fold - 1] if isinstance(e, tuple) else e


def glob_matrix(sd, t, fold):
    # the global embedding of a system's model(s) for fold k: seeds unit-normalised, concatenated, / √seeds; EF concatenates
    # ECHO and divides by √2 (exactly notebook 21's algebra, so its numbers come back)
    if sd["ft"] is None:
        if sd["enc"] == "BEATs" and sd["inp"] in ("CH0", "MWF", "N2N-CH0"):
            E = FZ[("BEATs", t, sd["inp"])]                           # notebook 20's frozen embeddings (the frozen references)
            return E[fold - 1] if isinstance(E, tuple) else E
        return np.asarray(model_pass(sd["enc"], None, sd["inp"], fold)[t]["P"], np.float32)
    Es = []
    for p in ft_models(sd, fold):
        D = np.load(p.with_name("emb_" + p.stem[3:] + ".npz"), allow_pickle=False)
        Es.append(D[f"{sd['kind']}__{t}"].astype(np.float32))
    E = np.concatenate([unit(e) for e in Es], 1) / math.sqrt(len(Es))
    if sd["ef"]:
        E = np.concatenate([E, unit(echo_of(t, sd["inp"], fold))], 1) / math.sqrt(2.0)
    return E


GM, BT = {}, {}


def glob_cached(sd, t, fold):
    k = (json.dumps({x: sd[x] for x in ("enc", "inp", "ft", "kind", "ef")}, sort_keys=True), t, fold)
    if k not in GM:
        GM[k] = glob_matrix(sd, t, fold)
    return GM[k]


def band_tensor(sd, t, fold, view):
    # (n, 8, D·seeds) on the device: each seed's band vectors unit-normalised per band, concatenated, / √seeds
    k = (json.dumps({x: sd[x] for x in ("enc", "inp", "ft")}, sort_keys=True), t, fold, view)
    if k not in BT:
        parts = ([model_pass(sd["enc"], None, sd["inp"], fold)[t][view]] if sd["ft"] is None
                 else [model_pass(sd["enc"], p, sd["inp"], fold)[t][view] for p in ft_models(sd, fold)])
        BT[k] = torch.cat([F.normalize(torch.from_numpy(np.asarray(a, np.float32)).to(DEVICE), dim=-1) for a in parts], -1) / math.sqrt(len(parts))
    return BT[k]


def echo_bands(t, inp, fold):
    k = ("ECHO", inp, t, fold)
    if k not in BT:
        e = echo_of(t, inp, fold); nb = ECHO_BANDS if e.shape[1] % ECHO_BANDS == 0 else 1
        BT[k] = F.normalize(torch.from_numpy(e.astype(np.float32).reshape(len(e), nb, -1)).to(DEVICE), dim=-1)
    return BT[k]


def comps_of(sd, t):
    # a system's weighted components for type t: (weight, "global" | "band", fold → matrix)
    if "parts" in sd:                                                 # a fusion: the mean of the parts' scores
        out = []
        for part in sd["parts"]:
            out += [(w / len(sd["parts"]), k, f) for w, k, f in comps_of(part, t)]
        return out
    if sd["scorer"] == "global":
        return [(1.0, "global", lambda fold, sd=sd: glob_cached(sd, t, fold))]
    share = 0.5 if sd["ef"] else 1.0
    vs = SCORER_VIEWS[sd["scorer"]]
    out = [(share / len(vs), "band", lambda fold, v=v, sd=sd: band_tensor(sd, t, fold, v)) for v in vs]
    if sd["ef"]:
        out.append((0.5, "band", lambda fold, sd=sd: echo_bands(t, sd["inp"], fold)))
    return out


def side(p, s):
    # the fold whose model (or denoiser) scores machine p in split s: its own fold for A, the other fold for B and B+n
    own = fold_of(p)
    return own if s == "A" else 3 - own


def score_of(comps, p, s, ref, te):
    k = side(p, s); out = 0.0
    for w, kind, f in comps:
        E = f(k)
        if kind == "global":
            out = out + w * ldn1(E[ref], E[te])
        else:
            r_, t_ = torch.from_numpy(ref).to(DEVICE), torch.from_numpy(te).to(DEVICE)
            out = out + w * band_ldn1(E[r_], E[t_]).cpu().numpy().astype(np.float64)
    return out


RES = {}


def skey(sd):
    return json.dumps(sd, sort_keys=True)


def res_sys(sd):
    k = skey(sd)
    if k not in RES:
        r = {}
        for t in TYPES:
            c = comps_of(sd, t)
            r[t] = evaluate_fn(lambda p, s, ref, te: score_of(c, p, s, ref, te), t)
            BT.clear()                                                # band tensors of one type at a time on the device
        RES[k] = r
    return RES[k]


def dc(sd):
    r = res_sys(sd)
    return S_of(r, MAIN), S_of(r, AUX)


def describe(sd):
    if "parts" in sd:
        return " + ".join("(" + describe(p) + ")" for p in sd["parts"])
    f = sd["ft"]
    who = f"{ENCODERS[sd['enc']]['name']} " + (f"fine-tuned ({f['src']}: {'+'.join(['R0'] + f['flags'])}, seeds {f['seeds']})" if f else "frozen")
    return f"{who} on {sd['inp']} · {sd['scorer']}" + (f" on {sd['kind']}" if sd["scorer"] == "global" else "") + (" + ECHO" if sd["ef"] else "")


def free_cache():
    BT.clear(); GM.clear(); gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

In [ ]:
# 13 · The program (Amendment 1): G0(b), G0(d) and G0(e) → encoder screen, frozen (N0, the selection) → the selected encoder, fine-tuned (N1, N2)
t_prog = time.time()
_N21 = {"ARR": sysd("BEATs", "MWF", kind="P", ft={"src": "N21", "track": "ARR", "flags": ["PD"], "seeds": [0]}),
        "MIC+": sysd("BEATs", "N2N-CH0"),
        "MIC": sysd("BEATs", "CH0", kind="F", ef=True, ft={"src": "N21", "track": "MIC", "flags": ["XC"], "seeds": [0, 1, 2]})}
N21_SYS = {tr: _N21[tr] for tr in TRACKS}
G0["b_n21"] = {}
for tr in TRACKS:
    d, c = dc(N21_SYS[tr]); G0["b_n21"][tr] = [d, c]
    print(f"G0(b) · notebook 21's {tr} system recomputed: development {d} · confirmation {c} (notebook 21: {N21_REF[tr]})")
_gap21 = max(abs(G0["b_n21"][tr][i] - N21_REF[tr][i]) for tr in TRACKS for i in (0, 1))
if not TEST and _gap21 > G0B_TOL:
    raise SystemExit("stop: G0(b) failed, notebook 21's files do not give its numbers")

# G0(d): notebook 21's final adapters, reloaded, reproduce its saved embeddings (first clips of every type)
G0["d"] = {}
for tr in ("ARR", "MIC"):
    if tr not in TRACKS:
        continue
    sd = N21_SYS[tr]
    for fold in (1, 2):
        for p in ft_models(sd, fold):
            st = torch.load(p, map_location="cpu", weights_only=False); load_lora(ENC["BEATs"], st["lora"])
            head = Head(768, CFG["emb_dim"]).to(DEVICE); head.load_state_dict(st["head"]); head.eval()
            D = np.load(p.with_name("emb_" + p.stem[3:] + ".npz"), allow_pickle=False); worst = 1.0
            for t in TYPES:
                rows = np.arange(min(G0C_CLIPS, len(META[t][0])))
                Pm, Fm, _ = run_pass(ENC["BEATs"], ST[t][stream_name(sd["inp"], fold)], head, rows, views=False)
                got = (Pm if sd["kind"] == "P" else Fm).astype(np.float64)
                worst = min(worst, float(np.min(np.sum(unit(got) * unit(D[f"{sd['kind']}__{t}"][rows].astype(np.float64)), 1))))
            G0["d"][p.stem] = round(worst, 6)
ENC["BEATs"].reset_lora()
print("G0(d) · notebook 21's adapters against its saved embeddings, smallest cosine per model:", G0["d"], f"(bar {G0D_COS})")
if not TEST and G0["d"] and min(G0["d"].values()) < G0D_COS:
    raise SystemExit("stop: G0(d) failed, notebook 21's adapters do not reproduce its embeddings")

# G0(e): notebook 22's final systems (stage 2's result), recomputed in this session; their band views come from notebook 21's
# adapters and notebook 20's denoisers
_N22 = {"ARR": {**_N21["ARR"], "scorer": "BEAM"}, "MIC+": {**_N21["MIC+"], "scorer": "BEAM"}, "MIC": dict(_N21["MIC"])}
N22_SYS = {tr: _N22[tr] for tr in TRACKS}
_f22 = OUT / "new_encoders_scoring.json"
if _f22.exists():
    _j22 = json.loads(_f22.read_text())["final"]
    G0["e_saved"] = {tr: bool(_j22[tr]["system"] == N22_SYS[tr] and tuple(_j22[tr]["dev_conf"]) == tuple(N22_REF[tr])) for tr in TRACKS if tr in _j22}
    print("notebook 22's saved final systems are the amendment's:", G0["e_saved"])
    if not TEST and not (len(G0["e_saved"]) == len(TRACKS) and all(G0["e_saved"].values())):
        raise SystemExit("stop: notebook 22's saved final systems are not the ones in the rules' amendment")
else:
    G0["e_saved"] = "notebook 22's JSON is not in Drive › P04_NewEncoders; the systems are the amendment's"
    print(G0["e_saved"])
G0["e"] = {}
for tr in TRACKS:
    d, c = dc(N22_SYS[tr]); G0["e"][tr] = [d, c]
    print(f"G0(e) · notebook 22's {tr} system recomputed: development {d} · confirmation {c} (notebook 22: {N22_REF[tr]})", flush=True)
_gap22 = max(abs(G0["e"][tr][i] - N22_REF[tr][i]) for tr in TRACKS for i in (0, 1))
if not TEST and _gap22 > G0E_TOL:
    raise SystemExit("stop: G0(e) failed, notebook 22's final systems do not come back in this session")
CUR = {tr: dict(N22_SYS[tr]) for tr in TRACKS}
FROZEN_BACKEND = "BEAM"                                               # stage 2's result on MIC+, in notebook 22
print("the frozen back-end for the encoder screen:", FROZEN_BACKEND)
STEPS = {tr: [] for tr in TRACKS}
REPORTED = {"screen": {}, "seed_spread_selected": {}}
free_cache()


def decide(step, tr, cand, ok=True, why=None):
    # the keep rule against the track's current system; the decision is recorded, and a kept candidate becomes the current system
    cur = CUR[tr]; rd, rc = dc(cur)
    rec = {"step": step, "candidate": describe(cand), "system": cand, "ref_dev": rd, "ref_conf": rc, "decidable": bool(ok)}
    if not ok:
        rec.update(kept=False, why=why or "not decidable"); STEPS[tr].append(rec)
        print(f"{tr} · {step}: not decidable ({rec['why']})", flush=True); return False
    d, c = dc(cand)
    kept = bool(d >= round(rd + BAR, 4) and c >= round(rc - BAR, 4))
    rec.update(dev=d, conf=c, bar=round(rd + BAR, 4), conf_bar=round(rc - BAR, 4), kept=kept)
    STEPS[tr].append(rec)
    if kept:
        CUR[tr] = cand
    print(f"{tr} · {step}: {describe(cand)}\n      development {d} against {rd} + {BAR} · confirmation {c} against {rc} − {BAR} → "
          f"{'kept' if kept else 'not kept'}", flush=True)
    return kept


# ---- stage 3 · encoder screen, frozen ----
SCREEN = {}
for ek in ENC:
    for tr in TRACKS:
        sd = sysd(ek, TRACK_IN[tr], scorer=FROZEN_BACKEND)
        SCREEN[f"{ek}|{tr}"] = dc(sd)
        for sc in ("global", "BEAM", "AdaBEAM"):
            REPORTED["screen"][f"{ek}|{tr}|{sc}"] = dc(sysd(ek, TRACK_IN[tr], scorer=sc))
        print(f"screen · {ek} on {tr} ({TRACK_IN[tr]}), {FROZEN_BACKEND}: development {SCREEN[f'{ek}|{tr}'][0]} · confirmation {SCREEN[f'{ek}|{tr}'][1]}", flush=True)
        free_cache()
NEW_OK = [k for k in NEW_ENCODERS if k in ENC]
if "MIC+" in TRACKS and NEW_OK:
    best = max(NEW_OK, key=lambda k: (SCREEN[f"{k}|MIC+"][0], -NEW_ENCODERS.index(k)))
    decide("N0", "MIC+", sysd(best, TRACK_IN["MIC+"], scorer=FROZEN_BACKEND))
ELIG = [k for k in NEW_OK if ENC[k].ok_lora]
SEL = max(ELIG, key=lambda k: (float(np.mean([SCREEN[f"{k}|{tr}"][0] for tr in TRACKS])), -NEW_ENCODERS.index(k))) if ELIG else None
SELECTION = {"eligible": ELIG, "mean_development": {k: round(float(np.mean([SCREEN[f"{k}|{tr}"][0] for tr in TRACKS])), 4) for k in NEW_OK},
             "selected": SEL}
if TEST and os.environ.get("P04_TEST_SEL") in ELIG:              # TEST only: walk stage 4 with a chosen encoder
    SEL = os.environ["P04_TEST_SEL"]; SELECTION["selected"] = SEL
SEL_FILE = OUT / "selection_22b.json"                              # a resumed session keeps the first session's selection
if SEL_FILE.exists():
    _prev = json.loads(SEL_FILE.read_text())["selected"]
    if _prev != SEL and _prev in ELIG:
        print(f"note: this session's screen selects {SEL}; an earlier session selected {_prev}, whose trainings are kept")
        SELECTION["selected_in_this_session"] = SEL; SEL = _prev; SELECTION["selected"] = SEL
elif SEL is not None:
    SEL_FILE.write_text(json.dumps(SELECTION))
print("selection for fine-tuning:", SELECTION, flush=True)


def encoders_in(sd):
    return set().union(*[encoders_in(p) for p in sd["parts"]]) if "parts" in sd else {sd["enc"]}


_keep = {SEL, "BEATs"}.union(*[encoders_in(CUR[tr]) for tr in TRACKS])
for d_ in VIEW_DIR.iterdir():
    if d_.name.split("__")[0] not in _keep:
        shutil.rmtree(d_, ignore_errors=True); PASS.pop(d_.name, None)
print(f"free local disk after the screen: {shutil.disk_usage(WORK).free / 1e9:.0f} GB")

# ---- stage 4 · the selected encoder, fine-tuned with each track's winning recipe ----
if SEL is not None:
    for tr in TRACKS:
        flags, seeds, kind = RECIPE[tr]
        ok = all([ensure(SEL, tr, TRACK_IN[tr], flags, s) for s in seeds])
        scorer = CUR[tr]["scorer"] if "parts" not in CUR[tr] else "global"
        new = sysd(SEL, TRACK_IN[tr], scorer=scorer, kind=kind, ef=(tr == "MIC"), ft={"src": "N22b", "track": tr, "flags": flags, "seeds": seeds})
        before = CUR[tr]
        n1 = decide("N1", tr, new, ok, "G1 failed")
        other = before if n1 else new
        decide("N2", tr, {"parts": [CUR[tr], other]}, ok, "G1 failed")
        if tr == "MIC" and ok:
            for s in seeds:
                REPORTED["seed_spread_selected"][str(s)] = dc(sysd(SEL, TRACK_IN[tr], kind="F", ft={"src": "N22b", "track": tr, "flags": flags, "seeds": [s]}))
        free_cache()
FINAL = {tr: CUR[tr] for tr in TRACKS}
for tr in TRACKS:
    d, c = dc(FINAL[tr])
    print(f"final · {tr}: {describe(FINAL[tr])}\n      development {d} · confirmation {c} (notebook 22: {N22_REF[tr]} · notebook 21: {N21_REF[tr]})")
print(f"program done · {(time.time() - t_prog) / 60:.0f} min in this session")

In [ ]:
# 14 · Reported numbers, save, two pictures, and one zip to download
import matplotlib
import matplotlib.pyplot as plt

REPORTED["T1_vs_published"], REPORTED["T2_arr_b10"], REPORTED["per_type_final"] = {}, {}, {}
for tr in TRACKS:
    fr, n22r, n21r = res_sys(FINAL[tr]), res_sys(N22_SYS[tr]), res_sys(N21_SYS[tr])
    REPORTED["T1_vs_published"][tr] = {t: {"published": PUBLISHED_M6[t], "final": fr[t]["A"]["avg"], "notebook_22": n22r[t]["A"]["avg"],
                                           "notebook_21": n21r[t]["A"]["avg"], "like_for_like": tr == "MIC"} for t in TYPES}
    REPORTED["per_type_final"][tr] = {t: {"final": [fr[t]["A"]["avg"], fr[t]["B+10"]["avg"]], "notebook_22": [n22r[t]["A"]["avg"], n22r[t]["B+10"]["avg"]],
                                          "notebook_21": [n21r[t]["A"]["avg"], n21r[t]["B+10"]["avg"]]} for t in TYPES}
if "ARR" in TRACKS:
    fa = res_sys(FINAL["ARR"])
    REPORTED["T2_arr_b10"] = {t: {"target_6dB": T2_TARGET[t], "final": fa[t]["B+10"]["avg"], "notebook_22": res_sys(N22_SYS["ARR"])[t]["B+10"]["avg"]}
                              for t in MAIN}
print("T1 · Split A at −6 dB against arXiv 2309.07500 (like for like only on MIC):")
for tr in TRACKS:
    print(f"  {tr:5s}", {t: (v["final"], v["published"]) for t, v in REPORTED["T1_vs_published"][tr].items()})
print("T2 · ARR B+10 against the 6 dB targets:", REPORTED["T2_arr_b10"])

RESULT = {"rules": RULES, "amendment": AMENDMENT, "program": PROGRAM, "date": time.strftime("%Y-%m-%d %H:%M"), "test_run": TEST,
          "meta": {"device": DEVICE, "gpu": GPU, "torch": torch.__version__, "checkpoint_sha256": CKPT_SHA, "code_sha256": code_sha,
                   "zips": {k: v[1:] for k, v in ZIPS.items()}, "xc_zips": {k: v[1:] for k, v in XZIPS.items()}, "cfg": CFG, "n2n_cfg": N2N_CFG,
                   "encoders": {k: {"name": e.name, "info": e.info, "layout": e.layout, "lora_ok": e.ok_lora, "step_seconds": e.step_s,
                                    "accepts_other_lengths": getattr(e, "varlen", None)} for k, e in ENC.items()},
                   "encoders_dropped": FAILED, "prep_seconds": PREP_S, "train_seconds": T_TRAIN, "pass_seconds": T_PASS,
                   "drive_free_gb": round(DRIVE_FREE_GB, 1)},
          "G0": G0, "GB": GBR, "peaks": SAT,
          "start": {tr: {"system": N22_SYS[tr], "describe": describe(N22_SYS[tr]), "dev_conf": G0["e"][tr], "notebook_22": N22_REF[tr]} for tr in TRACKS},
          "steps": STEPS, "frozen_backend": FROZEN_BACKEND, "screen": SCREEN,
          "selection": SELECTION, "final": {tr: {"system": FINAL[tr], "describe": describe(FINAL[tr]), "dev_conf": dc(FINAL[tr]),
                                                 "notebook_22": N22_REF[tr], "notebook_21": N21_REF[tr]} for tr in TRACKS},
          "reported": REPORTED, "xc": {t: {"per_machine": XMETA[t]["per_machine"], "rms_ratio_to_mimii_m6": XLEVEL[t]} for t in TYPES},
          "frozen": {tr: {"S": FROZEN_S[tr]} for tr in TRACKS},
          "results": {k: {"describe": describe(json.loads(k)), "dev_conf": [S_of(v, MAIN), S_of(v, AUX)], "res": v} for k, v in RES.items()},
          "G1": {tag: [lg["epochs"][0]["mean_loss"], lg["epochs"][-1]["mean_loss"]] for tag, lg in LOGS.items()},
          "lambda_pd": {tag: lg["lambda_pd"] for tag, lg in LOGS.items() if lg.get("lambda_pd") is not None},
          "training_logs": LOGS}
(OUT / "new_encoders_22b.json").write_text(json.dumps(RESULT, indent=1, default=float, ensure_ascii=False))

INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
fig, axes = plt.subplots(1, len(TRACKS), figsize=(5.4 * len(TRACKS), 4.3))
axes = np.atleast_1d(axes)
for ax, tr in zip(axes, TRACKS):
    st = [{"step": "start", "dev": G0["e"][tr][0], "conf": G0["e"][tr][1], "kept": True}] + [r for r in STEPS[tr] if r.get("dev") is not None]
    xs = np.arange(len(st))
    ax.plot(xs, [r["dev"] for r in st], "-o", color="#2a78d6", lw=2, ms=6, label="development (pumps, fans)")
    ax.plot(xs, [r["conf"] for r in st], "--s", color="#eb6834", lw=1.5, ms=5, label="confirmation (valves, slide rails)")
    for x_, r in zip(xs, st):
        if not r["kept"]:
            ax.plot([x_], [r["dev"]], "o", ms=9, mfc="white", mec="#2a78d6")
    ax.set_xticks(xs, [r["step"] for r in st], fontsize=8.5)
    ax.set_title(f"{tr} · {TRACK_IN[tr]} at −6 dB", loc="left", color=INK, fontsize=10.5)
    ax.grid(True, color=GRID, lw=0.6); ax.tick_params(colors=MUTED)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("score S = mean of (A + B+10)/2", color=MUTED)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, loc="lower center", ncol=2, fontsize=9)
fig.suptitle("Notebook 22b · each candidate against the current system (open circle: not kept; start: notebook 22)", fontsize=11)
fig.tight_layout(rect=(0, 0.08, 1, 0.95)); fig.savefig(OUT / "new_encoders_22b_steps.png", dpi=150); plt.show()

COL = {"ARR": "#eb6834", "MIC+": "#2a78d6", "MIC": "#1baf7a"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.3), sharey=True)
for ax, s in zip(axes, ("A", "B+10")):
    x = np.arange(len(TYPES)); w = 0.8 / (len(TRACKS) + 1)
    for j, tr in enumerate(TRACKS):
        fr = res_sys(FINAL[tr])
        ax.bar(x + (j - len(TRACKS) / 2) * w, [fr[t][s]["avg"] for t in TYPES], w * 0.92, color=COL.get(tr, "#8d8d8d"), label=f"{tr} final")
        ax.plot(x + (j - len(TRACKS) / 2) * w, [res_sys(N22_SYS[tr])[t][s]["avg"] for t in TYPES], "_", ms=14, mew=2, color=INK,
                label="notebook 22 (the start)" if j == 0 else None)
    if s == "A":
        ax.bar(x + (len(TRACKS) / 2) * w, [PUBLISHED_M6[t] for t in TYPES], w * 0.92, color="#c9c8c4", label="arXiv 2309.07500 (one microphone)")
    ax.set_xticks(x, TYPES); ax.set_ylim(0.5, 1.0); ax.grid(axis="y", color=GRID, lw=0.6); ax.set_axisbelow(True)
    ax.set_title("known machine (A)" if s == "A" else "new machine, 10 clips (B+10)", loc="left", color=INK, fontsize=10.5)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
axes[0].set_ylabel("average AUC at −6 dB", color=MUTED)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, loc="lower center", ncol=5, fontsize=9)
fig.tight_layout(rect=(0, 0.1, 1, 1)); fig.savefig(OUT / "new_encoders_22b_final.png", dpi=150); plt.show()

ZIP = OUT / "p04_n22b_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in ("new_encoders_22b.json", "new_encoders_22b_steps.png", "new_encoders_22b_final.png"):
        z.write(OUT / f, f)
print(f"saved {ZIP} · {(time.time() - T_START) / 60:.0f} min in this session")
if IN_COLAB:
    from google.colab import files
    files.download(str(ZIP))

## Send back
Send the zip that downloaded (`p04_n22b_results.zip`). It is also in Drive › `P04_NewEncoders`. The large files (adapters and embeddings) stay in Drive.